# Data aggregation

Builds one daily table, `data/processed/daily_panel.parquet` (one row per calendar day, 2021-01-01 to 2026-06-29), from all raw sources in `data/raw/`, plus a codebook that defines every column (`data/processed/daily_panel_codebook.csv`). The next step, `notebooks/02_lag_variables.ipynb`, adds lags, leads and moving averages to them; the analysis notebooks (`notebooks/03_1_cross_correlation.ipynb`, `notebooks/03_2_dlnm.ipynb`) load its output. Reports of this notebook go to `outputs/01_data_aggregation/`.

Each source section loads its data, plots it, and returns a DataFrame with **one row per calendar day** (index `date`) on the study calendar. Days a source does not cover are `NaN`. Section 4 compares the time frames and completeness of all sources before they are joined; section 7 writes the codebook; section 8 checks the raw files against the data manifest and records the run.

**All data stays local.** `data/` is gitignored. Outputs here are aggregated daily counts only; never print visit-level rows. Daily counts can be small (e.g. deaths in the ED): do not publish daily-level tables or charts without a small-number rule.

Preregistration: https://osf.io/6mcbd/overview. Changes relative to it are listed in `docs/deviations_log.md` (local only, not on GitHub).

## How to run

1. Create a virtual environment in `.venv` (project root) and select it as the notebook kernel. Python 3.12 is the tested version (`.python-version`); 3.14.0rc1 also works. The notebook finds the project root itself, so it runs from `notebooks/` or from the root.
2. Run **Install packages** once, restart the kernel, then **Run All**.
3. Put the MeteoGate API key in `.env` as `METEOALARM_API_KEY=...` (never committed). Without it, the official warnings section uses the cache only.

**Run time.** About 20 seconds when all caches exist. A first run downloads several sources (below); the MeteoAlarm archive is limited to 500 requests per hour, so a full first download (~445 requests) can take up to an hour if the limit was already used.

**Downloaded and cached by this notebook** (re-downloaded only if the cache file is missing, or the `*_FORCE_DOWNLOAD` flag is set):

| Source | Cache | Refresh |
|---|---|---|
| ERA5-Land / ERA5 reanalysis (Open-Meteo) | `data/raw/era5/` | delete the files or `ERA5_FORCE_DOWNLOAD = True` |
| Official warnings (MeteoAlarm, MeteoGate API) | `data/raw/meteoalarm/` | delete `sent_days_done.txt` and `maribor_messages.csv` |
| COVID-19 (Sledilnik API) | `data/raw/covid_sledilnik/` | `COVID_FORCE_DOWNLOAD = True` |
| COVID-19 restrictions (OxCGRT stringency index, Slovenia rows) | `data/raw/oxcgrt/` | delete the file |
| NDVI (Sentinel-2, Planetary Computer) and NUTS borders (GISCO) | `data/raw/ndvi/` | `NDVI_FORCE_DOWNLOAD = True` |
| Public holidays (Nager.Date), school holidays (OpenHolidays) | `data/raw/calendar/` | delete the yearly JSON files |
| Moon phases (US Naval Observatory) | `data/raw/moon/` | delete the yearly JSON files |

**Outputs** (gitignored). Analysis-ready data in `data/processed/`:

| File | Content |
|---|---|
| `daily_panel.parquet` | The daily table: one row per day, all outcomes and exposures |
| `daily_panel_codebook.csv` | Definition, unit, source, use (primary, sensitivity, …) and coverage of every column |

Reports in `outputs/01_data_aggregation/`:

| File | Content |
|---|---|
| `daily_panel_columns.csv` | First and last day with data and % missing per column |
| `dataset_coverage.csv` | Start, end and completeness of every source dataset |
| `holidays_crossref.csv` | Official holiday list vs Nager.Date, date by date |
| `warnings_validation.csv` | ARSO text warnings vs official warnings (sensitivity, PPV, κ) |
| `run_info.json` | Date, git commit, Python and package versions, manifest and panel checksums of the run |

**Reproducibility.** `requirements-lock.txt` pins every package version. `data/data_manifest.csv` (local only, like all data) lists every raw file with its size and SHA-256 checksum; the run stops if a raw file changed. After a deliberate data update, set `MANIFEST_UPDATE = True` once. The yearly ARSO weather exports are merged by `scripts/merge_arso_weather.py`.

## Data provenance

| Source | Folder | Provided by | How obtained | Status |
|---|---|---|---|---|
| HIS (ED visits) | `his/` | UKC Maribor | Extract from the hospital information system | [to fill: extracted by, date, query] |
| eTriaža | `etriaza/` | UKC Maribor | Yearly exports `PodrobnaStatistika_<year>.csv` | [to fill: exported by, date] |
| Weather | `arso_weather/` | ARSO | Yearly daily-climate exports for Maribor – letališče (`raw_yearly/`), merged by `scripts/merge_arso_weather.py` | Reproducible |
| Weather warnings (text) | `arso_warnings/` | ARSO | Warnings extracted from ARSO forecast texts | [missing: the extraction script is not in the repository] |
| Air quality | `arso_air/` | ARSO / EEA | EEA e-reporting files for station SI0059A; hourly extract `LPetravic_*.dsv` from ARSO | [to fill: request date] |
| ERVISS | `erviss/` | ECDC | Weekly respiratory surveillance files (see `erviss/README.md`) | [to fill: download date] |
| PCR testing | `PCR testing/` | NLZOH, Maribor | Daily positive tests, 2021-01 to 2024-02 | [to fill: provided by, date]; test totals requested |
| Pollen | `pollen/` | Pollen station Maribor | Daily counts 2021–2023 | [to fill: provider] |
| Google Trends | `google_trends/` | Google | Manual download, term "urgenca maribor", Slovenia, monthly | [to fill: download date] |
| SBI TOP | `sbi_top/` | Ljubljana Stock Exchange | Daily index history | [to fill: download date] |
| EMS dispatch | `EMS/` | EMS Maribor | Dispatch cards | [to fill: provided by, date] |
| Football fixtures | `NZS/` | NZS / MNZ Maribor | Fixture export | [to fill: provided by, date] |
| Hospital occupancy | `hospital capacity/` | UKC Maribor | Hourly occupancy by department | [to fill: provided by, date] |
| Public holidays (official) | `calendar/prazniki_uradni.csv` | Official calendar | | [to fill: source] |
| Downloaded sources | see the cache table above | ERA5 (Copernicus, via Open-Meteo, CC BY 4.0), MeteoAlarm, Sledilnik, Copernicus Sentinel-2, Eurostat GISCO, Nager.Date, OpenHolidays, US Naval Observatory | By this notebook | Reproducible (download dates in `run_info.json`) |

## Install packages

Installs the pinned versions in `requirements-lock.txt` (generated from `requirements.txt`) into the environment of the active kernel. Run it once in a new environment, then **restart the kernel** before running the rest. `%pip` (rather than `!pip`) makes sure the packages go into the kernel's own Python.

In [ ]:
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements-lock.txt").exists())
%pip install -q -r "{_root / 'requirements-lock.txt'}"

## Imports

In [ ]:
# Standard library
import hashlib
import io
import json
import os
import platform
import re
import subprocess
import time
from importlib.metadata import version as pkg_version
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd

# Plotting
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# Web downloads (COVID-19 API, NUTS borders)
import requests

# Geometry (NDVI areas)
from shapely.geometry import Point, Polygon, box, mapping, shape
from shapely.ops import unary_union

# Sentinel-2 NDVI (Microsoft Planetary Computer)
import planetary_computer
import pystac_client
from odc.geo.geom import Geometry
from odc.geo.xr import rasterize
from odc.stac import load as stac_load

## Configuration

In [ ]:
# Project root: the folder with requirements-lock.txt (works when run from notebooks/ or from the root)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements-lock.txt").exists())
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"            # analysis-ready data: daily panel and codebook
OUTPUTS = ROOT / "outputs" / "01_data_aggregation"  # reports of this notebook (coverage, validation, run record)
for d in (PROCESSED, OUTPUTS):
    d.mkdir(parents=True, exist_ok=True)

# Study period = HIS coverage (first and last complete day).
START = pd.Timestamp("2021-01-01")
END = pd.Timestamp("2026-06-29")
DAYS = pd.date_range(START, END, freq="D", name="date")

# Temporal alignment of low-frequency sources (see "Temporal alignment" below)
ERVISS_PUBLICATION_LAG_D = 5   # [assumption] days after an ISO week ends until ERVISS publishes it
GT_PUBLICATION_LAG_D = 1       # a monthly Google Trends value is complete the day after the month ends
SEWAGE_LAB_LAG_D = 3           # [assumption] days from wastewater sampling to a published result
NDVI_LAG_D = 1                 # Sentinel-2 L2A scenes are usually available within a day
RT_MAX_CARRY_D = 21            # weekly sources: carry the last published value at most this long
GT_MAX_CARRY_D = 45            # monthly sources: carry the last published value at most this long
SEWAGE_MAX_CARRY_D = 14        # carry a wastewater sample forward at most this long
NDVI_MAX_CARRY_D = 20          # carry an NDVI scene forward at most this long
NDVI_OUTLIER_DROP = 0.2        # drop a scene more than this below both neighbouring scenes (snow, cloud)

plt.rcParams.update({
    "figure.figsize": (12, 3.6), "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False, "font.size": 9,
})

### Input check

Before anything is written, every existing file under `data/raw/` is compared with `data/data_manifest.csv` (see section 8). A changed or new file stops the run here, so outputs are never overwritten from unexpected inputs. Missing files are allowed at this point only if the notebook downloads them (caches); section 8 checks the complete set again.

In [ ]:
MANIFEST = ROOT / "data" / "data_manifest.csv"   # local only (data/ is gitignored)
MANIFEST_UPDATE = False   # set True once after a deliberate data update
MANIFEST_IGNORE = {".DS_Store"}


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def raw_file_table() -> pd.DataFrame:
    files = sorted(p for p in RAW.rglob("*") if p.is_file() and p.name not in MANIFEST_IGNORE)
    return pd.DataFrame({"path": [p.relative_to(RAW).as_posix() for p in files],
                         "bytes": [p.stat().st_size for p in files],
                         "sha256": [sha256(p) for p in files]})


def compare_manifest(current: pd.DataFrame) -> dict[str, list]:
    old = pd.read_csv(MANIFEST)
    m = old.merge(current, on="path", how="outer", suffixes=("_manifest", "_now"), indicator=True)
    return {"added": m.loc[m["_merge"] == "right_only", "path"].tolist(),
            "missing": m.loc[m["_merge"] == "left_only", "path"].tolist(),
            "changed": m.loc[(m["_merge"] == "both") & (m["sha256_manifest"] != m["sha256_now"]), "path"].tolist()}


if MANIFEST.exists() and not MANIFEST_UPDATE:
    diff = compare_manifest(raw_file_table())
    if diff["added"] or diff["changed"]:
        raise RuntimeError(f"Raw data differ from {MANIFEST.name}: added {diff['added']}, changed {diff['changed']}. "
                           "If intended, set MANIFEST_UPDATE = True and run again.")
    print(f"Input check passed. Missing (to be downloaded, or absent): {diff['missing'] or 'none'}")
else:
    print("Input check skipped (no manifest yet, or MANIFEST_UPDATE = True)")

In [ ]:
def to_daily(df: pd.DataFrame) -> pd.DataFrame:
    """Reindex to the study calendar so every day is present (missing -> NaN)."""
    df = df.copy()
    df.index = pd.to_datetime(df.index).normalize()
    df.index.name = "date"
    dup = df.index[df.index.duplicated()]
    if len(dup):  # never drop data silently: a repeated date means the loader is wrong
        raise ValueError(f"to_daily: {len(dup)} repeated dates, e.g. {sorted(set(dup.strftime('%Y-%m-%d')))[:5]}")
    return df.sort_index().reindex(DAYS)


# Low-frequency inputs, kept for the look-ahead check in section 4
LOWFREQ_INPUTS: dict[str, dict] = {}


def _carry(df: pd.DataFrame, max_days: int) -> pd.DataFrame:
    """Daily series from values indexed by the day they become valid, carried forward at most max_days."""
    df = df[~df.index.duplicated(keep="last")].sort_index()
    full = pd.date_range(df.index.min(), df.index.max() + pd.Timedelta(days=max_days), freq="D")
    return df.reindex(full).ffill(limit=max_days)


def period_versions(values: pd.DataFrame, ref_start, ref_end, anchor, lag_days: int, max_carry_days: int) -> pd.DataFrame:
    """Daily versions of values observed per period (one row per week or month).

    main      the period's value on every day of the period (explanatory analyses)
    _rt       real time: the last value already published on that day (ref_end + lag_days), carried forward
    _interp   linear interpolation between period anchors (sensitivity analysis only; uses later periods)
    """
    ref_start, ref_end, anchor = (pd.DatetimeIndex(x) for x in (ref_start, ref_end, anchor))
    lengths = (ref_end - ref_start).days + 1
    days = np.concatenate([pd.date_range(a, b, freq="D").values for a, b in zip(ref_start, ref_end)])
    main = pd.DataFrame(np.repeat(values.to_numpy(dtype=float), lengths, axis=0), index=pd.DatetimeIndex(days), columns=values.columns)
    rt = _carry(pd.DataFrame(values.to_numpy(dtype=float), index=ref_end + pd.Timedelta(days=lag_days), columns=values.columns), max_carry_days)
    it = pd.DataFrame(values.to_numpy(dtype=float), index=anchor, columns=values.columns).sort_index()
    it = it.reindex(pd.date_range(it.index.min(), it.index.max(), freq="D")).interpolate(method="time", limit_area="inside")
    return to_daily(main).join(to_daily(rt).add_suffix("_rt")).join(to_daily(it).add_suffix("_interp"))


def sample_versions(values: pd.DataFrame, lag_days: int, max_carry_days: int) -> pd.DataFrame:
    """Daily versions of irregular point samples (index = sampling date).

    main       the last sample, carried forward from its sampling date for at most max_carry_days
    _rt        the same, but from the day the result is available (sampling date + lag_days)
    _interp    linear interpolation between samples (sensitivity analysis only; uses later samples)
    _age_days  days since the sample used by the main version
    """
    values = values.astype(float).sort_index()
    main = _carry(values, max_carry_days)
    rt = _carry(values.set_axis(values.index + pd.Timedelta(days=lag_days)), max_carry_days)
    it = values.reindex(pd.date_range(values.index.min(), values.index.max(), freq="D")).interpolate(method="time", limit_area="inside")
    last = _carry(pd.DataFrame({"d": values.index}, index=values.index), max_carry_days)["d"]
    age = pd.DataFrame({f"{c}_age_days": (last.index - last).dt.days for c in values.columns}, index=last.index)
    return to_daily(main).join(to_daily(rt).add_suffix("_rt")).join(to_daily(it).add_suffix("_interp")).join(to_daily(age))


def snake(s: str) -> str:
    s = s.lower().translate(str.maketrans("čšžćđ", "cszcd"))
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")


# Raw coverage of every dataset (before clipping to the study period), filled by the loaders
COVERAGE: dict[str, dict] = {}


def register(name, dates, group, resolution, provider, files):
    dates = pd.to_datetime(pd.Series(dates)).dropna()
    COVERAGE[name] = dict(group=group, provider=provider, resolution=resolution, files=files,
                          raw_start=dates.min().normalize(), raw_end=dates.max().normalize())


def study_span(ax):
    ax.axvspan(START, END, color="0.5", alpha=0.06, lw=0)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

### Temporal alignment of low-frequency sources

Weekly, monthly and irregularly sampled sources are put on the daily calendar without using later information. Each gets up to three versions:

| Column | Meaning | Use |
|---|---|---|
| *(no suffix)*, e.g. `ili_rate` | The value of the period the day belongs to (whole ISO week, whole month), or the last sample carried forward from its sampling date | **Main explanatory analyses** |
| `_rt` | Real time: the last value already **published** on that day (end of period or sampling date + publication lag), carried forward | **Forecasting** (module C) |
| `_interp` | Linear interpolation between periods or samples. Uses later values (look-ahead) | Sensitivity analysis only |
| `_age_days` | Days since the sample used by the main version (irregular samples) | Weighting, exclusion of stale values |

Publication lags and maximum carry-forward lengths are set in the configuration cell; lags marked [assumption] should be checked against the providers' release schedules. ERVISS also revises past weeks and only the latest version is available, so `_rt` columns are slightly better than what was truly known at the time; report this as a limitation. Section 4 checks that no `_rt` column depends on data published later.

Centred moving averages in the plotting cells are for display only and never enter the panel.

## 1. Outcomes: HIS linked to eTriaža

HIS (`his/POD.parquet`) holds one record per unit attendance: arrival time, ED unit, Manchester triage category (`KATE`), final diagnosis, hospital admission, and a pseudonymised patient ID (`PAC`). eTriaža (`etriaza/*.csv`) is joined to it by **`HIS.eTriazaId` = `eTriaža.id`** and adds arrival mode.

**ED units**

| ED unit | Stream |
|---|---|
| Enota za poškodbe (72), KPP | `injury` |
| Enota za bolezni (71) | `disease` |
| Internistična nujna pomoč (408), INP | `inp` |
| ZD – Hitri pregledi (HP) | `fasttrack` |

**Referral pathway.** Patients with an injury older than 24 h first go to Hitri pregledi (HP), which refers them on to KPP if needed. INP requires a referral, either from the patient's own GP or from HP. A referred patient therefore has two HIS records (HP, then KPP/INP), each with its own eTriaža triage. Counting records would count the same patient twice.

**Episodes.** Records of the same patient are linked into one **episode** when a record starts within `EPISODE_WINDOW_H` (6 h) of the discharge from the previous one. The referral gaps cluster within ~4 h and fall to a flat background after 6 h. Each episode is dated by its **first arrival**.

**Outcome columns**

| Columns | Unit | Definition |
|---|---|---|
| `arr_<stream>_<triage>` | episodes (patients) | Stream = unit where the episode **ended** (KPP → `injury`, INP → `inp`, Enota za bolezni → `disease`; `fasttrack` = seen in HP only). Triage = the **most urgent** category in the episode. Streams add up to `arr_all`. 5 streams × 6 triage levels = 30 series. |
| `ref_hp_<stream>` | referrals | HP → KPP / INP / Enota za bolezni within the episode window, dated by the HP arrival |
| `direct_<stream>` | episodes | Episodes whose first contact was that unit (ambulance, injury < 24 h, referral from own GP). GP referrals are not visible in HIS. |
| `att_<stream>` | records | Unit attendances as recorded, without linking (workload) |
| `adm_<stream>`, `adm_<stream>_pct` | episodes | Hospital admissions per day from each unit (the unit of the admitting record), and as % of that stream's episodes (`arr_<stream>_mts_all`). `adm_all`, `adm_all_pct` for all units. |
| `arr_medical_*`, `adm_medical*`, `ref_hp_medical`, `direct_medical`, `att_medical` | episodes | **Extra** combined stream: Enota za bolezni + INP. Not part of the stream sum. |
| `arr_dx_*`, `arr_died_in_ed`, `arr_age*`, `arr_ambulance` | episodes | Any record in the episode with that diagnosis/death; age and arrival mode from the first record |
| `arr_all_mts_all_w4h`, `arr_all_mts_all_w24h` | episodes | All episodes with a 4 h and 24 h window (sensitivity) |

Visits without a category (`KATE` 6) count only in `mts_all`. HP records no diagnoses.

**Structural break, 19 Nov 2022.** HP referrals drop overnight from ~20 to ~1 per day and never recover, and the HP share of attendances falls from 37% to 32%. Either the referral route changed or referred patients stopped getting a separate HIS record [to confirm with the ED]. Before the break, `att_*` and `arr_*` differ by ~4%; after it they nearly coincide. `hp_referral_era` = 1 before the break; use it as a break indicator in all stream-level models.

**Routing between Enota za bolezni and INP.** The two medical units split patients by urgency: Enota za bolezni takes almost all red/orange cases (≈ 30% admitted), INP mostly yellow/green (≈ 16% admitted). From 2021 to 2023 less urgent patients moved gradually from Enota za bolezni to INP (Enota za bolezni: 4,369 → ~2,800 episodes per quarter, orange 34% → 60%), while the two units combined stayed stable at ~6,000–7,000 per quarter with a stable triage mix. This is routing, not a change in patients [date and reason to confirm with the ED]. The combined `medical` stream is not affected by it and is the recommended main stream for H2/H3; the separate units remain available.

**Triage break, 7 June 2024.** The yellow (MTS 3) share fell from 56% to 31% in INP and from about 29% to 19% in Hitri pregledi over late May – late June 2024 (change point 2024-06-07), with green rising accordingly. Patient numbers did not change, and Enota za bolezni and KPP were barely affected. This points to a change in triage practice in the units that see lower-acuity patients [to confirm]. `triage_break_2024_06` = 1 from that date; include it in all triage-specific models.

**Data notes.** HIS times are local time (checked: no arrivals in the hour skipped at the spring clock change). Linkage HIS–eTriaža is complete wherever both exist; eTriaža ends on 2026-04-22. Hitri pregledi records **no diagnoses**, so the diagnosis counts (`arr_dx_*`) cover only the ~68% of episodes seen in KPP, INP or Enota za bolezni. Injuries (`arr_dx_injury_icd`) can serve as a negative-control outcome for air pollution and influenza, **not for weather**: injuries respond directly to ice, heat and outdoor activity.

**Gap check:** on a normal day HIS holds about 98% of the eTriaža records. Days below 85% are HIS extraction gaps (`his_gap` = 1), and all outcome columns on them are set to missing. eTriaža ends on 2026-04-22, so arrival mode is missing after that date.

In [ ]:
STREAM_MAP = {
    "Enota za poškodbe (72)": "injury",
    "Enota za bolezni (71)": "disease",
    "Internistična nujna pomoč (408)": "inp",
    "ZD - Hitri pregledi": "fasttrack",
}
STREAMS = ["injury", "disease", "inp", "fasttrack"]
SPECIALIST = ["injury", "disease", "inp"]   # units HP refers to
MTS = [1, 2, 3, 4, 5]
EPISODE_WINDOW_H = 6           # link records starting within 6 h of the previous discharge
SENSITIVITY_WINDOWS_H = [4, 24]
HIS_GAP_RATIO = 0.85           # HIS/eTriaža daily ratio below this = HIS extraction gap
HP_REFERRAL_BREAK = pd.Timestamp("2022-11-19")  # HP referrals drop from ~20/day to ~1/day (see chart)
TRIAGE_BREAK = pd.Timestamp("2024-06-07")   # yellow share INP 56% -> 31%, HP ~29% -> ~19% (change point; shift late May - late June 2024)
MEDICAL = ["disease", "inp"]                    # extra combined stream: Enota za bolezni + INP

# Diagnosis groups (ICD-10 prefix of the final diagnosis SDIA)
DX_GROUPS = {
    "F41": ("F41",),          # anxiety / panic (SBI TOP hypothesis)
    "R07": ("R07",),          # chest pain (SBI TOP hypothesis)
    "resp": ("J",),           # respiratory
    "circ": ("I",),           # circulatory
    "injury_icd": ("S", "T"), # injuries & poisoning (negative-control outcome)
}


def load_visits() -> tuple[pd.DataFrame, pd.Series]:
    his = pd.read_parquet(
        RAW / "his" / "POD.parquet",
        columns=["eTriazaId", "PAC", "CasPrihoda", "UC_DATOS", "EnotaUC_Naziv", "KATE", "SDIA",
                 "STAROST", "CASHOS", "UC_SMRT"],
    )
    etr = pd.concat(
        [pd.read_csv(f, sep="|", dtype=str, usecols=["id", "casprihoda", "nacinprihoda"])
         for f in sorted((RAW / "etriaza").glob("*.csv"))],
        ignore_index=True,
    )
    etr["id"] = etr["id"].astype("int64")
    etr["casprihoda"] = pd.to_datetime(etr["casprihoda"])
    assert etr["id"].is_unique and his["eTriazaId"].is_unique
    register("HIS", his["CasPrihoda"], "outcome", "per visit", "UKC Maribor", "his/POD.parquet")
    register("eTriaža", etr["casprihoda"], "outcome", "per visit", "UKC Maribor", "etriaza/*.csv")
    etr_daily = etr["casprihoda"].dt.normalize().value_counts().reindex(DAYS)

    v = his.merge(etr, left_on="eTriazaId", right_on="id", how="left", validate="one_to_one")
    v["date"] = v["CasPrihoda"].dt.normalize()
    v["stream"] = v["EnotaUC_Naziv"].map(STREAM_MAP)
    unmapped = v.loc[v["stream"].isna(), "EnotaUC_Naziv"].unique()
    assert len(unmapped) == 0, f"Unmapped ED units: {unmapped}"
    v["linked"] = v["id"].notna()
    v["mts"] = v["KATE"].where(v["KATE"].isin(MTS))  # 6 = no category
    in_etr = v["CasPrihoda"] <= etr["casprihoda"].max()
    print(f"HIS records: {len(v):,}; linked to eTriaža: {v.loc[in_etr, 'linked'].mean():.1%} of the records within eTriaža coverage "
          f"(until {etr['casprihoda'].max():%Y-%m-%d}); {int((~in_etr).sum()):,} records after that have no eTriaža record")
    # HIS times are local (Europe/Ljubljana): no arrival can fall in the hour skipped when clocks go forward
    spring_forward = pd.date_range(START, END, freq="D")
    spring_forward = spring_forward[(spring_forward.month == 3) & (spring_forward.day_of_week == 6) & (spring_forward.day >= 25)]
    in_gap = sum(((v["CasPrihoda"] >= d + pd.Timedelta(hours=2)) & (v["CasPrihoda"] < d + pd.Timedelta(hours=3))).sum() for d in spring_forward)
    assert in_gap == 0, f"{in_gap} arrivals in the skipped DST hour: HIS times are not local time"
    return v, etr_daily


def link_episodes(v: pd.DataFrame, window_h: float) -> pd.Series:
    """Episode number per record: a record starting within `window_h` hours after the
    previous record of the same patient ended belongs to the same episode."""
    v = v.sort_values(["PAC", "CasPrihoda"])
    prev_out = v.groupby("PAC")["UC_DATOS"].shift()
    gap = v["CasPrihoda"] - prev_out
    new = prev_out.isna() | (gap > pd.Timedelta(hours=window_h))
    return new.cumsum().reindex(v.index)


visits, etr_daily = load_visits()
visits = visits.sort_values(["PAC", "CasPrihoda"])
visits["episode"] = link_episodes(visits, EPISODE_WINDOW_H)
visits["prev_stream"] = visits.groupby("episode")["stream"].shift()
visits["is_hp_referral"] = (visits["prev_stream"] == "fasttrack") & visits["stream"].isin(SPECIALIST)

In [ ]:
def build_episodes(v: pd.DataFrame) -> pd.DataFrame:
    """One row per episode."""
    dx = v["SDIA"].fillna("")
    flags = pd.DataFrame({f"dx_{k}": dx.str.startswith(p) for k, p in DX_GROUPS.items()}, index=v.index)
    flags["admitted"] = v["CASHOS"].notna()
    flags["died"] = v["UC_SMRT"].eq("DA")
    flags["dx_missing"] = v["SDIA"].isna()
    refs = pd.DataFrame({f"ref_hp_{s}": v["is_hp_referral"] & (v["stream"] == s) for s in SPECIALIST}, index=v.index)
    adm_stream = v["stream"].where(v["CASHOS"].notna())  # unit of the admitting record
    g = v.assign(**flags, **refs, adm_stream=adm_stream).groupby("episode", sort=False)
    ep = pd.DataFrame({
        "date": g["date"].first(),
        "n_records": g.size(),
        "entry_stream": g["stream"].first(),
        "final_stream": g["stream"].last(),
        "mts": g["mts"].min(),                 # most urgent category in the episode
        "mts_first": g["mts"].first(),
        "age": g["STAROST"].first(),
        "arrival_mode": g["nacinprihoda"].first(),
        "n_hp_referrals": g["is_hp_referral"].sum(),
        "adm_stream": g["adm_stream"].last(),   # last non-missing = unit that admitted the patient
        **{c: g[c].any() for c in flags.columns},
        **{c: g[c].sum() for c in refs.columns},
    })
    ep["dx_missing"] = g["dx_missing"].all()
    return ep


episodes = build_episodes(visits)
episodes = episodes[(episodes["date"] >= START) & (episodes["date"] <= END)]
v_study = visits[visits["episode"].isin(episodes.index)]

summary = pd.Series({
    "HIS records": len(v_study),
    "episodes (patients)": len(episodes),
    "records merged into an earlier record": len(v_study) - len(episodes),
    "HP -> KPP referrals": int(episodes["ref_hp_injury"].sum()),
    "HP -> INP referrals": int(episodes["ref_hp_inp"].sum()),
    "HP -> Enota za bolezni referrals": int(episodes["ref_hp_disease"].sum()),
    "episodes with >2 records": int((episodes["n_records"] > 2).sum()),
})
summary.to_frame("count")

In [ ]:
def build_outcomes(v: pd.DataFrame, ep: pd.DataFrame) -> pd.DataFrame:
    by_day = lambda mask: ep[mask].groupby("date").size()
    out = {}
    # Episodes by final stream and most urgent triage
    for stream in ["all", *STREAMS]:
        s = pd.Series(True, index=ep.index) if stream == "all" else ep["final_stream"] == stream
        out[f"arr_{stream}_mts_all"] = by_day(s)
        for m in MTS:
            out[f"arr_{stream}_mts{m}"] = by_day(s & (ep["mts"] == m))
    # Referral pathway
    for s in SPECIALIST:
        out[f"ref_hp_{s}"] = ep.groupby("date")[f"ref_hp_{s}"].sum()
        out[f"direct_{s}"] = by_day(ep["entry_stream"] == s)
    # Episode characteristics
    for k in DX_GROUPS:
        out[f"arr_dx_{k}"] = by_day(ep[f"dx_{k}"])
    out["arr_dx_missing"] = by_day(ep["dx_missing"])
    # Hospital admissions by admitting unit (count and % of that stream's episodes)
    out["adm_all"] = by_day(ep["admitted"])
    for s in STREAMS:
        out[f"adm_{s}"] = by_day(ep["adm_stream"] == s)
    out["arr_died_in_ed"] = by_day(ep["died"])
    out["arr_age65plus"] = by_day(ep["age"] >= 65)
    out["arr_age_under18"] = by_day(ep["age"] < 18)
    out["arr_ambulance"] = by_day(ep["arrival_mode"].isin(["Nujno reševalno vozilo", "Reanimobil"]))
    # Unit attendances (records, unlinked)
    vr = v[(v["date"] >= START) & (v["date"] <= END)]
    out["att_all"] = vr.groupby("date").size()
    for s in STREAMS:
        out[f"att_{s}"] = vr[vr["stream"] == s].groupby("date").size()
    out["att_unlinked_etriaza"] = vr[~vr["linked"]].groupby("date").size()
    # Sensitivity: other episode windows (all arrivals only)
    for w in SENSITIVITY_WINDOWS_H:
        e = link_episodes(v, w)
        first = v.loc[~e.duplicated(), "date"]
        out[f"arr_all_mts_all_w{w}h"] = first[(first >= START) & (first <= END)].value_counts()

    daily = to_daily(pd.DataFrame(out)).fillna(0).astype(int)  # no visits = true zero

    # Extra combined stream: medical = Enota za bolezni + INP (not part of the stream sum)
    for suffix in ["mts_all", *[f"mts{m}" for m in MTS]]:
        daily[f"arr_medical_{suffix}"] = sum(daily[f"arr_{s}_{suffix}"] for s in MEDICAL)
    for prefix in ["adm", "ref_hp", "direct", "att"]:
        daily[f"{prefix}_medical"] = sum(daily[f"{prefix}_{s}"] for s in MEDICAL)

    # Reconciliation
    assert (daily[[f"arr_{s}_mts_all" for s in STREAMS]].sum(axis=1) == daily["arr_all_mts_all"]).all()
    assert (daily[[f"arr_all_mts{m}" for m in MTS]].sum(axis=1) <= daily["arr_all_mts_all"]).all()
    assert (daily[[f"att_{s}" for s in STREAMS]].sum(axis=1) == daily["att_all"]).all()
    assert daily["att_all"].sum() == len(vr)
    assert daily["arr_all_mts_all"].sum() == len(ep)
    assert daily["arr_all_mts_all_w24h"].sum() <= daily["arr_all_mts_all"].sum() <= daily["arr_all_mts_all_w4h"].sum()
    assert (daily[[f"adm_{s}" for s in STREAMS]].sum(axis=1) == daily["adm_all"]).all()

    # Extraction gaps: compare HIS records with eTriaža records
    ratio = daily["att_all"] / etr_daily
    daily["etr_count"] = etr_daily
    daily["hp_referral_era"] = (daily.index < HP_REFERRAL_BREAK).astype(int)
    daily["triage_break_2024_06"] = (daily.index >= TRIAGE_BREAK).astype(int)
    daily["his_gap"] = (ratio < HIS_GAP_RATIO).astype(int)
    etr_gap = etr_daily.isna() | (ratio > 1 / HIS_GAP_RATIO)
    daily = daily.astype(float)
    for s in ["all", *STREAMS, "medical"]:
        denom = daily[f"arr_{s}_mts_all"].replace(0, np.nan)
        daily[f"adm_{s}_pct"] = (100 * daily[f"adm_{s}"] / denom).round(2)
    outcome_cols = [c for c in daily.columns if c.startswith(("arr_", "ref_", "direct_", "att_", "adm_"))]
    daily.loc[daily["his_gap"] == 1, outcome_cols] = np.nan
    daily.loc[etr_gap, ["arr_ambulance", "att_unlinked_etriaza"]] = np.nan
    print("HIS gap days set to missing:", daily.index[daily["his_gap"] == 1].strftime("%Y-%m-%d").tolist())
    print("eTriaža gap days:", int(etr_gap.sum()))
    return daily


outcomes = build_outcomes(visits, episodes)
outcomes[["att_all", "arr_all_mts_all", "arr_all_mts_all_w4h", "arr_all_mts_all_w24h",
          *[f"arr_{s}_mts_all" for s in STREAMS], *[f"ref_hp_{s}" for s in SPECIALIST],
          *[f"adm_{s}" for s in ["all", *STREAMS]], *[f"adm_{s}_pct" for s in ["all", *STREAMS]]]].describe().round(1).T

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
colors = {"all": "black", "injury": "C3", "disease": "C0", "inp": "C4", "fasttrack": "C2"}

ax = axes[0]
for s in ["all", *STREAMS]:
    ax.plot(outcomes.index, outcomes[f"arr_{s}_mts_all"].rolling(7, center=True).mean(), lw=1.1, color=colors[s], label=s)
ax.plot(outcomes.index, outcomes["att_all"].rolling(7, center=True).mean(), lw=0.9, color="0.5", ls="--", label="attendances (records)")
for d in outcomes.index[outcomes["his_gap"] == 1]:
    ax.axvline(d, color="red", lw=0.6, ls=":")
ax.set_title("ED episodes per day by final stream (7-day mean); dashed = unit attendances; dotted red = HIS gap days")
ax.legend(ncol=6, loc="upper left")

ax = axes[1]
monthly = outcomes[[f"arr_all_mts{m}" for m in MTS]].resample("MS").sum()
share = monthly.div(monthly.sum(axis=1), axis=0)
mts_colors = ["#d62728", "#ff7f0e", "#e5c100", "#2ca02c", "#1f77b4"]
ax.stackplot(share.index, share.T.values, colors=mts_colors, labels=[f"MTS {m}" for m in MTS], step="post")
ax.set_ylim(0, 1)
ax.set_title("Manchester triage mix per month (most urgent category per episode)")
ax.legend(ncol=5, loc="lower left", bbox_to_anchor=(0, 1.08))

ax = axes[2]
weekly = outcomes[["adm_all", "arr_ambulance", "arr_dx_resp", "arr_dx_R07", "arr_dx_circ"]].resample("W").sum(min_count=7)  # full weeks only
for c in weekly:
    ax.plot(weekly.index, weekly[c], lw=1, label=c.replace("arr_", ""))
ax.set_title("Weekly episodes: admissions, ambulance arrivals, diagnosis groups")
ax.legend(ncol=5, loc="upper left")
study_span(ax)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [2.2, 1]})
ax = axes[0]
for s, col in [("injury", "C3"), ("inp", "C4"), ("disease", "C0")]:
    ax.plot(outcomes.index, outcomes[f"ref_hp_{s}"].rolling(7, center=True).mean(), color=col, lw=1.2, label=f"HP → {s}")
ax.axvline(HP_REFERRAL_BREAK, color="0.3", ls="--", lw=0.8)
ax.text(HP_REFERRAL_BREAK, ax.get_ylim()[1] * 0.92, f" {HP_REFERRAL_BREAK:%Y-%m-%d}", fontsize=8, color="0.3")
ax.set_title("Referrals from Hitri pregledi per day (7-day mean)"); ax.legend(loc="upper left")
study_span(ax)

ax = axes[1]
tot = outcomes[[*[f"direct_{s}" for s in SPECIALIST], *[f"ref_hp_{s}" for s in SPECIALIST]]].sum()
x = np.arange(len(SPECIALIST))
ax.bar(x, [tot[f"direct_{s}"] for s in SPECIALIST], color="0.6", label="direct")
ax.bar(x, [tot[f"ref_hp_{s}"] for s in SPECIALIST], bottom=[tot[f"direct_{s}"] for s in SPECIALIST], color="C1", label="referred by HP")
for i, s in enumerate(SPECIALIST):
    share = tot[f"ref_hp_{s}"] / (tot[f"direct_{s}"] + tot[f"ref_hp_{s}"])
    ax.text(i, tot[f"direct_{s}"] + tot[f"ref_hp_{s}"], f"{share:.0%} via HP", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x, ["KPP (injury)", "INP", "Enota za bolezni"])
ax.set_title("Route into each unit, whole period"); ax.legend(loc="upper right")
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6.5), sharex=True)
colors["medical"] = "C9"
for s in ["all", *STREAMS, "medical"]:
    axes[0].plot(outcomes.index, outcomes[f"adm_{s}"].rolling(7, center=True).mean(), lw=1.1, color=colors[s], label=s, ls="--" if s == "medical" else "-")
    num = outcomes[f"adm_{s}"].rolling(28, center=True, min_periods=21).sum()
    den = outcomes[f"arr_{s}_mts_all"].rolling(28, center=True, min_periods=21).sum()
    axes[1].plot(outcomes.index, 100 * num / den, lw=1.1, color=colors[s], label=s, ls="--" if s == "medical" else "-")
axes[0].set_title("Hospital admissions per day by admitting unit (7-day mean)"); axes[0].legend(ncol=6, loc="upper left")
axes[1].set_title("Admission rate by unit, % of that stream's patients (28-day pooled)"); axes[1].legend(ncol=5, loc="upper left")
axes[1].axvline(HP_REFERRAL_BREAK, color="0.3", ls="--", lw=0.8)
study_span(axes[1]); fig.tight_layout()

In [ ]:
units = [("disease", "Enota za bolezni"), ("inp", "INP"), ("medical", "Medical = Enota za bolezni + INP"),
         ("injury", "KPP (injury)"), ("fasttrack", "Hitri pregledi")]
q = outcomes.resample("QS").sum(min_count=1)
fig, axes = plt.subplots(len(units), 1, figsize=(12, 2.3 * len(units)), sharex=True)
for ax, (s, label) in zip(axes, units):
    share = pd.DataFrame({f"MTS {m}": q[f"arr_{s}_mts{m}"] / q[f"arr_{s}_mts_all"] for m in MTS})
    ax.stackplot(share.index, share.T.values, colors=mts_colors, step="post", labels=share.columns)
    n = q[f"arr_{s}_mts_all"]
    ax2 = ax.twinx(); ax2.step(n.index, n, where="post", color="black", lw=1.1); ax2.grid(False)
    ax2.set_ylabel("episodes / quarter", fontsize=7); ax2.set_ylim(0, n.max() * 1.15)
    for d in [HP_REFERRAL_BREAK, TRIAGE_BREAK]:
        ax.axvline(d, color="white", ls="--", lw=1)
    ax.set_ylim(0, 1); ax.set_ylabel("share"); ax.set_title(f"{label}: triage mix (colours) and episodes per quarter (black line)", fontsize=9)
axes[0].legend(ncol=5, loc="lower left", bbox_to_anchor=(0, 1.25), fontsize=8)
axes[-1].xaxis.set_major_locator(mdates.YearLocator()); axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle("Triage mix per unit by quarter; dashed lines = HP referral break (2022-11-19) and triage break (2024-06-07)", fontsize=10)
fig.tight_layout()

## 2. Exposures

### 2.1 Weather (ARSO, Maribor – letališče)

Daily station data. The file has no humidity, pressure, or wind speed.

**How ARSO defines the daily values** (ARSO, *Razlaga meteoroloških spremenljivk*). Fixed observation times are in CET (UTC+1), i.e. 08:00, 15:00 and 22:00 local time in summer.

| Column | ARSO definition |
|---|---|
| `temp_mean` | (T7 + T14 + 2·T21) / 4 from the 07, 14 and 21 h CET readings at 2 m (Kämtz). Not a 24-hour mean, so it differs slightly from the ERA5 daily mean |
| `temp_max`, `temp_min` | Highest and lowest temperature at 2 m of the day |
| `precip` | 24-hour total read at 07:00 CET, **dated by the reading day**: the raw value of day *d* fell from 07:00 CET on *d* − 1 to 07:00 CET on *d* |
| `snow_new` | New snow in the past 24 hours, measured at 07:00 CET: same convention as precipitation |
| `snow_depth` | Total snow depth measured at 07:00 CET (a morning state) |
| `sunshine` | Sunshine duration of the day |
| `cloud` | Mean of the cloud cover estimated at 07, 14 and 21 h CET |
| `wx_storm` | Thunderstorm (*nevihta*): lightning seen and thunder heard |
| `wx_thunder`, `wx_lightning`, `wx_hail` | Thunder (*grmenje*), lightning (*bliskanje*), hail (*toča*, ice 5–50 mm) observed |
| `wx_gale` | Gale (*viharni veter*): wind ≥ 8 Beaufort (≥ 17.2 m/s). Since 2000 observers also use automatic-station data at night, so more days are recorded than before |

**Precipitation and new snow are shifted back one day**, so that day *d* holds the 24 hours from 07:00 CET on *d* to 07:00 CET on *d* + 1 (the raw value of *d* + 1), which mostly falls on day *d*. Without the shift, 17 of the 24 hours would belong to the previous day. An independent check against ERA5 hourly precipitation (`scripts/check_arso_precip_dating.py`) agrees: ARSO's daily value matches the reading-day window (Pearson r 0.77) far better than the calendar day (0.35). Snow depth is a morning state and is not shifted; the other variables refer to the calendar day.

**Temperature events** (0/1, fixed thresholds), an objective alternative to heat and cold warnings: `hot_day` (Tmax ≥ 30 °C), `very_hot_day` (Tmax ≥ 35 °C), `tropical_night` (Tmin ≥ 20 °C), `heatwave` (part of a run of ≥ 3 hot days), `frost_day` (Tmin < 0 °C), `ice_day` (Tmax < 0 °C), `cold_day` (Tmin ≤ −10 °C).

In [ ]:
HEATWAVE_MIN_DAYS = 3


def load_arso_weather() -> pd.DataFrame:
    f = RAW / "arso_weather" / "maribor_weather_2019_2026_merged.csv"
    w = pd.read_csv(f, parse_dates=["date"])
    assert w["station_id"].nunique() == 1
    register("Weather (ARSO)", w["date"], "exposure", "daily", "ARSO", "arso_weather/" + f.name)
    w = w.set_index("date").rename(columns={
        "t_avg_c": "temp_mean", "t_max_c": "temp_max", "t_min_c": "temp_min", "t_range_c": "temp_range",
        "precip_mm": "precip", "snow_depth_cm": "snow_depth", "new_snow_cm": "snow_new",
        "sunshine_h": "sunshine", "cloud_pct": "cloud",
    })
    flags = ["storm", "thunder", "lightning", "hail", "gale"]
    w[flags] = w[flags].astype(float)
    w = w.rename(columns={f: f"wx_{f}" for f in flags})
    keep = ["temp_mean", "temp_max", "temp_min", "temp_range", "precip", "snow_depth", "snow_new",
            "sunshine", "cloud", *[f"wx_{f}" for f in flags]]
    w = w[keep].asfreq("D")
    # ARSO dates the 07:00 CET 24-hour totals (precipitation, new snow) by the reading day: day d = mostly d-1.
    # Shift back one day so that day d holds 07:00 CET d -> 07:00 CET d+1 (see markdown above).
    w[["precip", "snow_new"]] = w[["precip", "snow_new"]].shift(-1)
    # Temperature events (fixed thresholds; computed on the full series so runs at the edges are correct)
    tmax, tmin = w["temp_max"], w["temp_min"]
    event = lambda cond, x: cond.astype(float).where(x.notna())
    w["hot_day"] = event(tmax >= 30, tmax)            # Tmax >= 30 °C
    w["very_hot_day"] = event(tmax >= 35, tmax)       # Tmax >= 35 °C
    w["tropical_night"] = event(tmin >= 20, tmin)     # Tmin >= 20 °C
    run = (tmax >= 30).groupby((tmax < 30).cumsum()).transform("sum")
    w["heatwave"] = event((tmax >= 30) & (run >= HEATWAVE_MIN_DAYS), tmax)  # >= 3 consecutive hot days
    w["frost_day"] = event(tmin < 0, tmin)            # Tmin < 0 °C
    w["ice_day"] = event(tmax < 0, tmax)              # Tmax < 0 °C
    w["cold_day"] = event(tmin <= -10, tmin)          # Tmin <= -10 °C
    return to_daily(w)


arso_weather = load_arso_weather()

In [ ]:
w = arso_weather
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].fill_between(w.index, w["temp_min"], w["temp_max"], color="C1", alpha=0.25, lw=0, label="min–max")
axes[0].plot(w.index, w["temp_mean"], color="C1", lw=0.8, label="mean")
axes[0].set_title("Air temperature (°C)"); axes[0].legend(loc="upper left")
axes[1].bar(w.index, w["precip"], width=1, color="C0")
axes[1].set_title("Precipitation (mm/day)")
axes[2].bar(w.index, w["snow_depth"], width=1, color="C9", label="snow depth (cm)")
axes[2].plot(w.index, w["sunshine"].rolling(7, center=True).mean() * 3, color="C8", lw=0.8, label="sunshine h ×3 (7-day mean)")
axes[2].set_title("Snow and sunshine"); axes[2].legend(loc="upper left")
study_span(axes[2]); fig.tight_layout()

#### 2.1.1 Reanalysis weather (ERA5-Land and ERA5)

The ARSO station has no humidity, wind or pressure, so these come from the Copernicus reanalyses, downloaded through the Open-Meteo archive API (no key; data licence CC BY 4.0, attribution: Copernicus Climate Change Service / ECMWF, Open-Meteo):

- **ERA5-Land** (0.1° grid, cell 46.6 N 15.6 E, covering Maribor): air temperature, relative humidity, dew point.
- **ERA5** (0.25° grid, cell 46.5 N 15.75 E): wind speed at 10 m, surface pressure. ERA5-Land in Open-Meteo has no wind or pressure.

Hourly values are requested in UTC and aggregated to **local** days (Europe/Ljubljana; 23- and 25-hour days at the clock changes); a day needs ≥ 20 hours. **Apparent temperature** is computed per hour with Steadman's formula as used by the Australian Bureau of Meteorology (shade version), a common choice in heat-health studies:

AT = T + 0.33·e − 0.70·ws − 4.00, with water-vapour pressure e = RH/100 · 6.105 · exp(17.27·T / (237.7 + T)) [hPa], ws in m/s.

Feels-like temperature is defined below.

ERA5-Land temperature is compared with the ARSO station below; the run stops if the correlation of daily mean temperature falls below 0.95. **The station remains the primary temperature exposure**; the reanalysis adds humidity, wind, pressure and apparent temperature, and serves as a sensitivity analysis for temperature.

**Feels-like temperature** (`era5_feels_like_mean/max/min`) follows the US National Weather Service definition used by weather apps, computed per hour:

- **heat index** (Rothfusz regression with the NWS adjustments) when T ≥ 26.7 °C (80 °F) and relative humidity ≥ 40%;
- **wind chill** (NWS / Environment Canada formula: 13.12 + 0.6215·T − 11.37·V^0.16 + 0.3965·T·V^0.16, V in km/h at 10 m) when T ≤ 10 °C and V > 4.8 km/h;
- otherwise the air temperature.

It complements the Steadman apparent temperature above, which applies one formula at all temperatures.

In [ ]:
def heat_index_c(t_c: pd.Series, rh: pd.Series) -> pd.Series:
    """NWS heat index (Rothfusz regression with adjustments), °C in and out."""
    t = t_c * 9 / 5 + 32
    simple = 0.5 * (t + 61.0 + (t - 68.0) * 1.2 + rh * 0.094)
    hi = (-42.379 + 2.04901523 * t + 10.14333127 * rh - 0.22475541 * t * rh - 0.00683783 * t ** 2
          - 0.05481717 * rh ** 2 + 0.00122874 * t ** 2 * rh + 0.00085282 * t * rh ** 2 - 0.00000199 * t ** 2 * rh ** 2)
    low_rh = (rh < 13) & (t >= 80) & (t <= 112)
    hi = hi - np.where(low_rh, (13 - rh) / 4 * np.sqrt(np.clip((17 - np.abs(t - 95)) / 17, 0, None)), 0)
    high_rh = (rh > 85) & (t >= 80) & (t <= 87)
    hi = hi + np.where(high_rh, (rh - 85) / 10 * (87 - t) / 5, 0)
    hi = np.where((simple + t) / 2 < 80, simple, hi)  # NWS: use the simple formula when the result is below 80 °F
    return (pd.Series(hi, index=t_c.index) - 32) * 5 / 9


def wind_chill_c(t_c: pd.Series, wind_kmh: pd.Series) -> pd.Series:
    v = wind_kmh ** 0.16
    return 13.12 + 0.6215 * t_c - 11.37 * v + 0.3965 * t_c * v


def feels_like(t_c: pd.Series, rh: pd.Series, wind_ms: pd.Series) -> pd.Series:
    wind_kmh = wind_ms * 3.6
    out = t_c.copy()
    hot = (t_c >= 26.7) & (rh >= 40)
    cold = (t_c <= 10) & (wind_kmh > 4.8)
    out[hot] = heat_index_c(t_c, rh)[hot]
    out[cold] = wind_chill_c(t_c, wind_kmh)[cold]
    return out


# Checks against published NWS values: heat index 90 °F at 60% RH = 100 °F; wind chill -10 °C at 20 km/h = -17.9 °C
assert abs(heat_index_c(pd.Series([(90 - 32) * 5 / 9]), pd.Series([60.0])).iloc[0] * 9 / 5 + 32 - 100) < 1
assert abs(wind_chill_c(pd.Series([-10.0]), pd.Series([20.0])).iloc[0] - (-17.9)) < 0.1

In [ ]:
ERA5_API = "https://archive-api.open-meteo.com/v1/archive"
ERA5_DIR = RAW / "era5"
ERA5_DIR.mkdir(parents=True, exist_ok=True)
ERA5_FORCE_DOWNLOAD = False
ERA5_POINT = dict(latitude=46.5541, longitude=15.6447)  # UKC Maribor; the API returns the containing grid cell
ERA5_REQUESTS = {
    "era5_land": dict(models="era5_land", hourly="temperature_2m,relative_humidity_2m,dew_point_2m"),
    "era5": dict(models="era5", hourly="wind_speed_10m,surface_pressure", wind_speed_unit="ms"),
}
ERA5_MIN_HOURS = 20


def download_era5() -> dict[str, pd.DataFrame]:
    out = {}
    for name, extra in ERA5_REQUESTS.items():
        f = ERA5_DIR / f"{name}_hourly_utc.csv"
        meta_f = ERA5_DIR / f"{name}_request.json"
        params = ERA5_POINT | extra | dict(start_date=(START - pd.Timedelta(days=2)).strftime("%Y-%m-%d"),
                                           end_date=(END + pd.Timedelta(days=2)).strftime("%Y-%m-%d"), timezone="GMT")
        stale = not meta_f.exists() or json.loads(meta_f.read_text())["params"] != params  # e.g. START/END changed
        if ERA5_FORCE_DOWNLOAD or not f.exists() or stale:
            r = requests.get(ERA5_API, params=params, timeout=300)
            r.raise_for_status()
            d = r.json()
            pd.DataFrame(d["hourly"]).to_csv(f, index=False)
            meta_f.write_text(json.dumps({"params": params, "grid_cell": [d["latitude"], d["longitude"]], "elevation": d["elevation"],
                                          "units": d["hourly_units"], "downloaded": pd.Timestamp.now().isoformat(timespec="seconds")}, indent=1))
            print(f"downloaded {name}: grid cell {d['latitude']:.2f} N {d['longitude']:.2f} E")
        out[name] = pd.read_csv(f, parse_dates=["time"])
    return out


def apparent_temperature(t: pd.Series, rh: pd.Series, ws: pd.Series) -> pd.Series:
    """Steadman apparent temperature (Australian Bureau of Meteorology, shade): T in °C, RH in %, ws in m/s."""
    e = rh / 100 * 6.105 * np.exp(17.27 * t / (237.7 + t))
    return t + 0.33 * e - 0.70 * ws - 4.00


def load_era5() -> pd.DataFrame:
    raw = download_era5()
    h = raw["era5_land"].merge(raw["era5"], on="time", how="inner", validate="one_to_one")
    h["at"] = apparent_temperature(h["temperature_2m"], h["relative_humidity_2m"], h["wind_speed_10m"])
    h["fl"] = feels_like(h["temperature_2m"], h["relative_humidity_2m"], h["wind_speed_10m"])
    h["date"] = h["time"].dt.tz_localize("UTC").dt.tz_convert("Europe/Ljubljana").dt.tz_localize(None).dt.normalize()
    register("Reanalysis weather (ERA5-Land / ERA5)", h["time"], "exposure", "hourly", "Copernicus ERA5 via Open-Meteo", "era5/*_hourly_utc.csv")
    g = h.groupby("date")
    daily = pd.DataFrame({
        "era5_temp_mean": g["temperature_2m"].mean(),
        "era5_temp_max": g["temperature_2m"].max(),
        "era5_temp_min": g["temperature_2m"].min(),
        "era5_rh_mean": g["relative_humidity_2m"].mean(),
        "era5_dewpoint_mean": g["dew_point_2m"].mean(),
        "era5_apparent_temp_mean": g["at"].mean(),
        "era5_apparent_temp_max": g["at"].max(),
        "era5_feels_like_mean": g["fl"].mean(),
        "era5_feels_like_max": g["fl"].max(),
        "era5_feels_like_min": g["fl"].min(),
        "era5_wind_mean": g["wind_speed_10m"].mean(),
        "era5_wind_max": g["wind_speed_10m"].max(),
        "era5_pressure_mean": g["surface_pressure"].mean(),
    }).where(g.size() >= ERA5_MIN_HOURS)  # incomplete days -> missing
    return to_daily(daily.round(2))


era5 = load_era5()
check = arso_weather[["temp_mean", "temp_max"]].join(era5[["era5_temp_mean", "era5_temp_max"]]).dropna()
r_mean = check["temp_mean"].corr(check["era5_temp_mean"])
print(f"ERA5-Land vs ARSO station: r(daily mean) = {r_mean:.3f}, mean difference {(check['era5_temp_mean'] - check['temp_mean']).mean():+.2f} °C; "
      f"r(daily max) = {check['temp_max'].corr(check['era5_temp_max']):.3f}, mean difference {(check['era5_temp_max'] - check['temp_max']).mean():+.2f} °C")
assert r_mean > 0.95, "ERA5-Land temperature does not match the station: check the grid cell and time zone"
era5.describe().round(1).T[["mean", "min", "max"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [1, 2.4]})
ax = axes[0]
ax.scatter(check["temp_mean"], check["era5_temp_mean"], s=3, alpha=0.3, color="C1")
lim = [check[["temp_mean", "era5_temp_mean"]].min().min() - 1, check[["temp_mean", "era5_temp_mean"]].max().max() + 1]
ax.plot(lim, lim, color="0.3", lw=0.8, ls="--")
ax.set_xlabel("ARSO station, daily mean (°C)"); ax.set_ylabel("ERA5-Land, daily mean (°C)")
ax.set_title(f"Temperature: station vs reanalysis (r = {r_mean:.3f})")
ax = axes[1]
e7 = era5.rolling(7, center=True, min_periods=4).mean()
ax.plot(e7.index, e7["era5_temp_mean"], color="C1", lw=0.9, label="air temperature (°C)")
ax.plot(e7.index, e7["era5_apparent_temp_mean"], color="C3", lw=0.9, label="apparent temperature (°C)")
ax.plot(e7.index, e7["era5_feels_like_mean"], color="C4", lw=0.9, ls="--", label="feels-like (NWS, °C)")
ax2 = ax.twinx(); ax2.plot(e7.index, e7["era5_rh_mean"], color="C0", lw=0.7, alpha=0.6, label="relative humidity (%, right)")
ax2.grid(False); ax2.set_ylim(30, 105)
ax.set_title("ERA5 daily means (7-day mean)")
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, ncol=3, loc="lower left", bbox_to_anchor=(0, 1.08))
study_span(ax); fig.tight_layout()

### 2.2 Weather warnings

Two sources with different roles:
- **Official warnings (MeteoAlarm, 2.2.2)**: an **exposure** for analysis, but only from April 2025 (`MA_VALID_FROM`).
- **ARSO text warnings (2.2.1)**: **exploratory**; they capture only about a quarter of the official warning days (validation in 2.2.3).

Warnings are partly a consequence of the weather itself (a mediator). In models, adjust for the weather variables: the warning coefficient then estimates the behavioural effect of being warned.

#### 2.2.1 ARSO text warnings (whole period)

Text warnings with a type, area and validity period, taken from ARSO's forecast texts. They carry **no warning level**, 41% of the dates since 2021 were assumed from the issue date, and about half match Maribor only through vague areas. Three versions:

| Columns | Rule |
|---|---|
| `warn_<type>`, `warn_any` | **Broad** (sensitivity): any warning whose area could include Maribor (`MARIBOR_AREAS_BROAD`) |
| `warn_strict_<type>`, `warn_strict_any` | **Strict**: explicit date **and** an area that clearly includes Maribor (`MARIBOR_AREAS_STRICT`) |
| `warn_date_assumed` | 1 if every broad warning covering that day has an assumed or approximate date |

For heat and cold, prefer the temperature-based events in the weather section (`hot_day`, `heatwave`, …): only one heat warning passes the strict rule.

In [ ]:
MARIBOR_AREAS_STRICT = {"cela Slovenija", "severovzhodna Slovenija", "vzhodna Slovenija", "Štajerska"}
MARIBOR_AREAS_BROAD = MARIBOR_AREAS_STRICT | {"Slovenija (ni določeno)", "nižine", "notranjost države", "severna Slovenija", "Pohorje"}
WARN_TYPES_SKIP = {"sea", "avalanche", "burja"}  # not applicable to Maribor


def load_arso_warnings() -> pd.DataFrame:
    f = RAW / "arso_warnings" / "arso_opozorila_1995_2026.csv"
    w = pd.read_csv(f, encoding="utf-8-sig", parse_dates=["start_date", "end_date"])
    register("Weather warnings (ARSO text)", pd.concat([w["start_date"], w["end_date"]]), "exploratory", "per warning", "ARSO", "arso_warnings/" + f.name)
    areas = w["area"].str.split("; ").apply(set)
    w["broad"] = areas.apply(lambda a: bool(MARIBOR_AREAS_BROAD & a))
    w["strict"] = areas.apply(lambda a: bool(MARIBOR_AREAS_STRICT & a)) & (w["date_basis"] == "explicit")
    w["dated"] = w["date_basis"] == "explicit"
    w = w[w["broad"] & (w["end_date"] >= START)]
    rows = []
    for _, r in w.iterrows():
        for t in r["warning_type"].split("; "):
            if t not in WARN_TYPES_SKIP:
                for d in pd.date_range(r["start_date"], r["end_date"], freq="D"):
                    rows.append((d, t, r["strict"], r["dated"]))
    long = pd.DataFrame(rows, columns=["date", "type", "strict", "dated"])
    broad = pd.crosstab(long["date"], long["type"]).clip(upper=1).add_prefix("warn_")
    broad["warn_any"] = broad.max(axis=1)
    s = long[long["strict"]]
    strict = pd.crosstab(s["date"], s["type"]).clip(upper=1).add_prefix("warn_strict_")
    strict["warn_strict_any"] = strict.max(axis=1)
    assumed = (~long.groupby("date")["dated"].any()).astype(int).rename("warn_date_assumed")
    out = to_daily(broad).join(to_daily(strict)).join(to_daily(assumed.to_frame()))
    return out.fillna(0).astype(int)  # days without a warning are 0


arso_warnings = load_arso_warnings()
arso_warnings.filter(regex="_any$|assumed").sum().rename("days").to_frame()

In [ ]:
types = arso_warnings.filter(regex="^warn_(?!strict|any|date)").sum().sort_values().index
fig, ax = plt.subplots(figsize=(12, 0.35 * len(types) + 1))
for i, t in enumerate(types):
    d = arso_warnings.index[arso_warnings[t] == 1]
    ax.scatter(d, np.full(len(d), i), marker="|", s=80, color="#ef9a9a")
    st = t.replace("warn_", "warn_strict_")
    if st in arso_warnings:
        d = arso_warnings.index[arso_warnings[st] == 1]
        ax.scatter(d, np.full(len(d), i), marker="|", s=80, color="#b71c1c")
labels = [f"{t.replace('warn_', '')} ({int(arso_warnings[t].sum())} d; strict {int(arso_warnings.get(t.replace('warn_', 'warn_strict_'), pd.Series(0)).sum())} d)" for t in types]
ax.set_yticks(range(len(types)), labels)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#ef9a9a", "#b71c1c"]]
ax.legend(handles, ["broad", "strict"], ncol=2, loc="lower left", bbox_to_anchor=(0, 1.0))
ax.set_title("ARSO text warnings covering Maribor", loc="right")
study_span(ax); fig.tight_layout()

#### 2.2.2 Official warnings (MeteoAlarm, from April 2025)

Official ARSO warnings in CAP format, from the MeteoAlarm archive through the MeteoGate API. The archive only starts on **2025-04-12** (although its metadata says 2019), so these cover the last ~14 months of the study period. They are an **exposure** for analyses restricted to that window, and they validate the text warnings (2.2.3).

- The API key is read from `.env` (`METEOALARM_API_KEY`, never committed). Without a key the section uses the cache only.
- The API returns warnings by the day they were **sent** (max 24 h per request), so the archive is downloaded day by day and cached in `data/raw/meteoalarm/`. Only parsed fields are cached: API responses contain the key in their links.
- A message counts for Maribor if its region polygon contains UKC Maribor (region *Slovenija / severovzhod*). It is active from `onset` until `expires`, or until it was superseded by a newer message; *All clear* messages and level 1 (green) are ignored.
- Daily columns (from `MA_VALID_FROM`, earlier days are missing): `ma_warn_level` = highest level active in Maribor that day (0 none, 2 yellow, 3 orange, 4 red), `ma_warn_any` (yellow or higher), `ma_warn_orange_plus` (orange or red), and `ma_warn_<hazard>` = highest level per hazard.

In [ ]:
MA_API = "https://api.meteogate.eu/warnings/collections/warnings/locations/SI"
MA_ARCHIVE_START = pd.Timestamp("2025-04-12")                 # first day with archived messages
MA_VALID_FROM = MA_ARCHIVE_START + pd.Timedelta(days=5)       # warnings are sent up to ~5 days before onset
MA_DIR = RAW / "meteoalarm"
MA_DIR.mkdir(parents=True, exist_ok=True)
MARIBOR_POINT = Point(15.6447, 46.5541)                        # UKC Maribor (lon, lat)
HAZARD_BY_EVENT = {  # Slovene event prefix -> hazard
    "visoka temperatura": "high_temp", "nizka temperatura": "low_temp", "nevihte": "thunderstorm",
    "veter": "wind", "dež": "rain", "sneg": "snow_ice", "poledica": "snow_ice", "požarna ogroženost": "forest_fire",
    "megla": "fog", "poplave": "flooding", "obalni dogodki": "coastal", "plazovi": "avalanche", "snežni plazovi": "avalanche",
}


def read_env(name: str, path: Path = ROOT / ".env") -> str | None:
    if name in os.environ:
        return os.environ[name]
    if path.exists():
        for line in path.read_text().splitlines():
            if line.startswith(name + "="):
                return line.split("=", 1)[1].strip()
    return None


MA_PAUSE_S = 0.5  # pause between requests; the gateway allows 500 requests per hour (HTTP 429 beyond that)


def hazard_of(event: str) -> str:
    """Hazard from the Slovene event text, e.g. 'Nevihte - zmerna ogroženost' -> 'thunderstorm'.
    Longest prefix first, so 'snežni plazovi' wins over shorter matches."""
    e = str(event).lower()
    for k in sorted(HAZARD_BY_EVENT, key=len, reverse=True):
        if e.startswith(k):
            return HAZARD_BY_EVENT[k]
    return snake(e.split(" - ")[0])


def get_json(url: str, params: dict | None = None, tries: int = 8):
    """GET with retries and back-off. Errors never show the URL or parameters (they contain the API key)."""
    for attempt in range(tries):
        time.sleep(MA_PAUSE_S)
        try:
            r = requests.get(url, params=params, timeout=180)
            if r.status_code == 204:
                return None
            if r.status_code == 200:
                return r.json()
            err = f"HTTP {r.status_code}"
            if r.status_code == 429:  # rate limit (500 requests/hour): wait until the window resets
                wait = float(r.headers.get("X-RateLimit-Reset") or r.headers.get("Retry-After") or 60 * 2 ** attempt)
                wait = min(wait + 5, 3700)
                print(f"  rate limit reached; waiting {wait / 60:.0f} min until it resets")
                time.sleep(wait)
                continue
        except Exception as e:  # noqa: BLE001
            err = type(e).__name__
        time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"MeteoAlarm request failed after {tries} tries ({err}); rerun the cell to resume")


def parse_cap(cap: dict, feature_props: dict) -> list[dict]:
    rows = []
    for info in cap.get("info", []):
        if info.get("language") != "sl":
            continue
        params = {p["valueName"]: p["value"] for p in info.get("parameter", [])}
        level = int(str(params.get("awareness_level", "0")).split(";")[0])
        event = info.get("event", "")
        hazard = hazard_of(event)
        for area in info.get("area", []):
            for poly in area.get("polygon", []):
                pts = [tuple(map(float, p.split(",")))[::-1] for p in poly.split()]  # CAP: "lat,lon"
                if Polygon(pts).contains(MARIBOR_POINT):
                    rows.append(dict(
                        alert_id=feature_props["alertId"], sent=cap.get("sent"), msg_type=cap.get("msgType"),
                        response_type=(info.get("responseType") or [None])[0], event=event, hazard=hazard, level=level,
                        onset=info.get("onset") or info.get("effective"), expires=info.get("expires"),
                        superseded_at=feature_props.get("supersededAt"), area=area.get("areaDesc"),
                    ))
    return rows


def download_meteoalarm(key: str, last_day: pd.Timestamp) -> None:
    done_f, msg_f = MA_DIR / "sent_days_done.txt", MA_DIR / "maribor_messages.csv"
    done = set(done_f.read_text().split()) if done_f.exists() else set()
    days = [d for d in pd.date_range(MA_ARCHIVE_START, last_day, freq="D") if d.strftime("%Y-%m-%d") not in done]
    print(f"MeteoAlarm: {len(days)} days to download")
    for i, day in enumerate(days):
        ds = day.strftime("%Y-%m-%d")
        page = get_json(MA_API, params={"datetime": f"{ds}T00:00:00Z/{ds}T23:59:59Z", "apikey": key})
        rows, seen = [], set()
        for f in (page or {}).get("features", []):
            lon = [c[0] for c in f["geometry"]["coordinates"][0]]
            lat = [c[1] for c in f["geometry"]["coordinates"][0]]
            aid = f["properties"]["alertId"]
            if aid in seen or not (min(lon) <= MARIBOR_POINT.x <= max(lon) and min(lat) <= MARIBOR_POINT.y <= max(lat)):
                continue
            seen.add(aid)
            url = next(l["href"] for l in f["links"] if l["rel"] == "json")
            rows += parse_cap(get_json(url), f["properties"])
        if rows:
            pd.DataFrame(rows).to_csv(msg_f, mode="a", header=not msg_f.exists(), index=False)
        with open(done_f, "a") as fh:
            fh.write(ds + "\n")
        if i % 50 == 0:
            print(f"  {ds}: {len(rows)} Maribor messages")


ma_key = read_env("METEOALARM_API_KEY")
if ma_key:
    download_meteoalarm(ma_key, END)
elif (MA_DIR / "maribor_messages.csv").exists():
    print("No METEOALARM_API_KEY in .env: using the cache only")
else:
    print("WARNING: no METEOALARM_API_KEY in .env and no cache in data/raw/meteoalarm/. The official-warning columns will be "
          "missing and the final manifest check will stop the run. Add the key to .env, or copy data/raw/meteoalarm/ "
          "from a colleague's checkout.")

In [ ]:
def load_meteoalarm() -> pd.DataFrame | None:
    f = MA_DIR / "maribor_messages.csv"
    if not f.exists():
        return None
    m = pd.read_csv(f).drop_duplicates()
    m["hazard"] = m["event"].map(hazard_of)  # re-derive, so mapping changes apply to cached rows too
    for c in ["onset", "expires", "superseded_at", "sent"]:
        m[c] = pd.to_datetime(m[c], utc=True, format="ISO8601").dt.tz_convert("Europe/Ljubljana").dt.tz_localize(None)
    m = m[(m["level"] >= 2) & (m["response_type"] != "AllClear") & (m["msg_type"] != "Cancel")]
    m["end"] = m[["expires", "superseded_at"]].min(axis=1)
    m = m[m["end"] > m["onset"]]
    lead_days = (m["onset"] - m["sent"]).dt.total_seconds() / 86400
    print(f"Lead time from sending to onset: median {lead_days.median():.1f} days, max {lead_days.max():.1f} days")
    assert lead_days.max() < (MA_VALID_FROM - MA_ARCHIVE_START).days, "warnings are sent earlier than MA_VALID_FROM allows: move it later"
    register("Weather warnings (MeteoAlarm)", pd.concat([m["onset"], m["end"]]), "exposure", "per message", "MeteoAlarm / ARSO", "meteoalarm/maribor_messages.csv")
    rows = []
    for _, r in m.iterrows():  # a day counts if the warning is active at any time that day
        for d in pd.date_range(r["onset"].normalize(), (r["end"] - pd.Timedelta(seconds=1)).normalize(), freq="D"):
            rows.append((d, r["hazard"], r["level"]))
    long = pd.DataFrame(rows, columns=["date", "hazard", "level"])
    wide = long.pivot_table(index="date", columns="hazard", values="level", aggfunc="max").add_prefix("ma_warn_")
    wide["ma_warn_level"] = long.groupby("date")["level"].max()
    daily = to_daily(wide).fillna(0.0)
    daily["ma_warn_any"] = (daily["ma_warn_level"] >= 2).astype(float)          # yellow or higher
    daily["ma_warn_orange_plus"] = (daily["ma_warn_level"] >= 3).astype(float)  # orange or red
    daily.loc[daily.index < MA_VALID_FROM] = np.nan  # before the archive: unknown, not "no warning"
    global meteoalarm_messages
    meteoalarm_messages = m
    return daily


meteoalarm = load_meteoalarm()
if meteoalarm is not None:
    print(f"{len(meteoalarm_messages):,} active Maribor messages; days with a warning by level:")
    display(meteoalarm["ma_warn_level"].value_counts().sort_index().rename({0: "none", 2: "yellow", 3: "orange", 4: "red"}).to_frame("days"))

In [ ]:
if meteoalarm is not None:
    ma = meteoalarm.dropna(how="all")
    hz = ma.drop(columns=["ma_warn_level", "ma_warn_any", "ma_warn_orange_plus"]).sum().sort_values().index
    level_color = {2: "#f9a825", 3: "#ef6c00", 4: "#c62828"}
    fig, ax = plt.subplots(figsize=(12, 0.4 * len(hz) + 1.2))
    for i, h in enumerate(hz):
        for lv, col in level_color.items():
            d = ma.index[ma[h] == lv]
            ax.scatter(d, np.full(len(d), i), marker="|", s=110, color=col)
    ax.set_yticks(range(len(hz)), [f"{h.replace('ma_warn_', '')} ({int((ma[h] > 0).sum())} d)" for h in hz])
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in level_color.values()]
    ax.legend(handles, ["yellow", "orange", "red"], ncol=3, loc="lower left", bbox_to_anchor=(0, 1.0), title="warning level")
    ax.set_title("Official warnings active in Maribor (MeteoAlarm), by hazard and level", loc="right")
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.tight_layout()

#### 2.2.3 Validation: ARSO text warnings vs official warnings

Over the overlap (from `MA_VALID_FROM` to the end of the study period), each day is compared: does the text-based variable flag a warning on days with an official warning (level ≥ 2) in Maribor? **Sensitivity** = share of official warning days also flagged by the text data; **PPV** = share of text-flagged days that had an official warning; **κ** = Cohen's kappa.

In [ ]:
HAZARD_MATCH = {  # ARSO text type(s) -> MeteoAlarm hazard
    "heat": ["high_temp"], "cold": ["low_temp"], "thunderstorm": ["thunderstorm"], "hail": ["thunderstorm"],
    "wind": ["wind"], "heavy_rain": ["rain", "flooding"], "flooding": ["rain", "flooding"],
    "snow": ["snow_ice"], "ice_glaze": ["snow_ice"], "fire_danger": ["forest_fire"], "any": None,
}


def agreement(a: pd.Series, b: pd.Series) -> dict:
    a, b = a.astype(bool), b.astype(bool)
    n, both = len(a), (a & b).sum()
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return dict(official_days=int(b.sum()), text_days=int(a.sum()), both=int(both),
                sensitivity=both / b.sum() if b.sum() else np.nan, ppv=both / a.sum() if a.sum() else np.nan,
                kappa=(po - pe) / (1 - pe) if pe < 1 else np.nan)


if meteoalarm is not None:
    window = meteoalarm["ma_warn_level"].dropna().index
    res = []
    for text_type, hazards in HAZARD_MATCH.items():
        if hazards is None:
            official = meteoalarm.loc[window, "ma_warn_level"] >= 2
        else:
            cols = [f"ma_warn_{h}" for h in hazards if f"ma_warn_{h}" in meteoalarm]
            official = (meteoalarm.loc[window, cols] >= 2).any(axis=1) if cols else pd.Series(False, index=window)
        for version, prefix in [("broad", "warn_"), ("strict", "warn_strict_")]:
            col = f"{prefix}{text_type}"
            if col in arso_warnings:
                res.append(dict(text_type=text_type, version=version, **agreement(arso_warnings.loc[window, col], official)))
    validation = pd.DataFrame(res)
    validation.to_csv(OUTPUTS / "warnings_validation.csv", index=False)
    display(validation.round(2))

### 2.3 Air quality (ARSO), Maribor stations only

Only PM measurements from Maribor are used:

| Source | Station | Pollutants | Columns |
|---|---|---|---|
| EEA e-reporting `SI0059A` (verified `e1a` + unverified `utd` 2026) | Maribor Vrbanski plato (urban background; EEA code of the MB Vrbanski station) | PM10, PM2.5, NO2, O3 | `pm10`, `pm25`, `no2`, `o3` |
| ARSO hourly extract (`LPetravic_*.dsv`), installation **E407** | MB Titova (Maribor Center, traffic) | PM10, PM2.5 | `pm10_titova`, `pm25_titova` |
| same, installation **E408** | MB Vrbanski (urban background): the **same station** as SI0059A, ARSO's hourly data stream (r = 0.98 with the EEA daily values) | PM10, PM2.5 | `pm10_vrbanski`, `pm25_vrbanski` |

The `.dsv` extract also contains 20 stations elsewhere in Slovenia. These are dropped. Installation codes come from ARSO's station list (`arso.gov.si/xml/zrak/ones_zrak_urni_podatki_zadnji.xml`).

All air-quality data are on a fixed UTC+1 clock (EU reporting convention; every day has 24 hours, also at the clock changes), so in summer a "day" runs 01:00–01:00 local time. `pm10` / `pm25` (SI0059A) are the primary series. Only values with `Validity` > 0 are used, and a daily mean from hourly data needs ≥ 18 valid hours. In 2026 SI0059A reports PM only as hourly, unverified data from a different sampling point (`_101`); those days are flagged in `pm_unverified_instrument`. O3 is the maximum daily 8-hour running mean, following the EU rule.

In [ ]:
EEA_POLLUTANT = {5: "pm10", 6001: "pm25", 7: "o3", 8: "no2"}
MARIBOR_INSTALLATIONS = {"E407": "titova", "E408": "vrbanski"}


def read_eea(pattern: str) -> pd.DataFrame:
    files = sorted((RAW / "arso_air").glob(pattern))
    d = pd.concat([pd.read_parquet(f, columns=["Pollutant", "Start", "End", "Value", "Validity"]) for f in files])
    d["Value"] = pd.to_numeric(d["Value"].astype(str), errors="coerce")
    return d[(d["Validity"] > 0) & (d["Value"] >= 0)]


def hourly_to_daily_mean(h: pd.DataFrame) -> pd.Series:
    g = h.groupby(h["Start"].dt.normalize())["Value"]
    return g.mean().where(g.count() >= 18)


def o3_max8h(h: pd.DataFrame) -> pd.Series:
    s = h.set_index("End")["Value"].sort_index()
    s = s[~s.index.duplicated()].asfreq("h")
    roll = s.rolling(8, min_periods=6).mean()
    day = (roll.index - pd.Timedelta(minutes=1)).normalize()  # windows ending 01:00..24:00 belong to that day
    g = roll.groupby(day)
    return g.max().where(g.count() >= 18)


def load_arso_air() -> pd.DataFrame:
    day = read_eea("e1a_day_*.parquet")
    hour = pd.concat([read_eea("e1a_hour_*.parquet"), read_eea("utd_hour_*.parquet")])
    out = {}
    for pcode, name in EEA_POLLUTANT.items():
        h = hour[hour["Pollutant"] == pcode]
        d = day[day["Pollutant"] == pcode]
        if name == "o3":
            s = o3_max8h(h)
        else:
            verified = pd.Series(d["Value"].values, index=d["Start"].dt.normalize())
            s = verified.combine_first(hourly_to_daily_mean(h))
            if name in ("pm10", "pm25"):
                out[f"_{name}_from_hourly"] = s.notna() & ~s.index.isin(verified.dropna().index)
        out[name] = s
        register(f"Air SI0059A {name.upper()}", pd.concat([d["Start"], h["Start"]]), "exposure", "hourly/daily", "ARSO / EEA", "arso_air/*SI0059A*.parquet")
    air = pd.DataFrame(out)
    # Days whose PM10 or PM2.5 come from the hourly, unverified data of sampling point _101 (a different instrument, 2026)
    air["pm_unverified_instrument"] = (air.pop("_pm10_from_hourly").fillna(False) | air.pop("_pm25_from_hourly").fillna(False)).astype(int)

    for f, pol in [("LPetravic_PM10.dsv", "pm10"), ("LPetravic_PM2_5.dsv", "pm25")]:
        n = pd.read_csv(RAW / "arso_air" / f, sep=";", decimal=",")
        n = n[n["INSTALACIJA"].isin(MARIBOR_INSTALLATIONS) & (n["VELJAVNOST_1H"] == 1)]
        n["t"] = pd.to_datetime(n["DATUM_ZAJEMA"], format="%d.%m.%Y %H:%M")
        for inst, site in MARIBOR_INSTALLATIONS.items():
            x = n[n["INSTALACIJA"] == inst]
            g = x.groupby(x["t"].dt.normalize())["POV_1H"]
            air[f"{pol}_{site}"] = g.mean().where(g.count() >= 18)
            register(f"Air {inst} MB {site.capitalize()} {pol.upper()}", x["t"], "exposure", "hourly", "ARSO", f"arso_air/{f}")
    return to_daily(air)


arso_air = load_arso_air()
arso_air.corr().round(2)

In [ ]:
a = arso_air.rolling(7, center=True, min_periods=4).mean()
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for c, col in [("pm10", "black"), ("pm10_titova", "C3"), ("pm10_vrbanski", "C0")]:
    axes[0].plot(a.index, a[c], lw=0.9, color=col, label=c)
axes[0].axhline(50, color="0.4", ls="--", lw=0.7); axes[0].text(START, 52, "EU daily limit 50 µg/m³", fontsize=8, color="0.4")
axes[0].set_title("PM10, Maribor stations (µg/m³, 7-day mean)"); axes[0].legend(ncol=3, loc="upper right")
for c, col in [("pm25", "black"), ("pm25_titova", "C3"), ("pm25_vrbanski", "C0")]:
    axes[1].plot(a.index, a[c], lw=0.9, color=col, label=c)
axes[1].set_title("PM2.5, Maribor stations (µg/m³, 7-day mean)"); axes[1].legend(ncol=3, loc="upper right")
axes[2].plot(a.index, a["o3"], color="C1", lw=0.9, label="O3 max 8-h")
axes[2].plot(a.index, a["no2"], color="C5", lw=0.9, label="NO2")
axes[2].set_title("O3 and NO2, SI0059A (µg/m³, 7-day mean)"); axes[2].legend(loc="upper right")
study_span(axes[2]); fig.tight_layout()

#### European Air Quality Index (EAQI)

The **revised European Air Quality Index** of the European Environment Agency (bands aligned with the WHO 2021 guidelines; https://airindex.eea.europa.eu/AQI/index.html, ETC HE Report 2024/17). The index is defined **hourly**: each pollutant gets a level from its hourly concentration, and the worst pollutant sets the hour's level (1 good, 2 fair, 3 moderate, 4 poor, 5 very poor, 6 extremely poor). Concentrations are rounded to whole µg/m³ before banding, as the bands are published in whole numbers.

| Level | PM2.5 | PM10 | O3 | NO2 |
|---|---|---|---|---|
| 1 Good | 0–5 | 0–15 | 0–60 | 0–10 |
| 2 Fair | 6–15 | 16–45 | 61–100 | 11–25 |
| 3 Moderate | 16–50 | 46–120 | 101–120 | 26–60 |
| 4 Poor | 51–90 | 121–195 | 121–160 | 61–100 |
| 5 Very poor | 91–140 | 196–270 | 161–180 | 101–150 |
| 6 Extremely poor | > 140 | > 270 | > 180 | > 150 |

SO2 is not measured at these stations and is left out. **Hourly data used:** NO2 and O3 from station SI0059A; PM10 and PM2.5 from **E408 MB Vrbanski** for 2021–2025 (SI0059A reports PM only as daily values until 2025; E408 is the same station in ARSO's hourly data, r = 0.98) and from SI0059A (hourly, unverified) for 2026. All on the same UTC+1 clock.

Daily columns: `aqi_max` (worst hour), `aqi_mean` (mean hourly level), `aqi_hours_poor` (hours at level 4 or worse), `aqi_n_hours` (hours with an index). A day needs ≥ 18 hours with an index.

In [ ]:
EAQI_BANDS = {  # upper bounds (µg/m³, whole numbers) of levels 1-5; above the last = level 6
    "pm25": [5, 15, 50, 90, 140],
    "pm10": [15, 45, 120, 195, 270],
    "o3": [60, 100, 120, 160, 180],
    "no2": [10, 25, 60, 100, 150],
}
EAQI_LABELS = {1: "good", 2: "fair", 3: "moderate", 4: "poor", 5: "very poor", 6: "extremely poor"}
AQI_MIN_HOURS = 18


def eaqi_level(conc: pd.Series, pollutant: str) -> pd.Series:
    bounds = EAQI_BANDS[pollutant]
    c = np.floor(conc + 0.5)  # round half up to whole µg/m³
    level = pd.Series(np.searchsorted(bounds, c.to_numpy(), side="left") + 1, index=conc.index, dtype=float)
    return level.where(conc.notna())


def build_aqi() -> pd.DataFrame:
    hour = pd.concat([read_eea("e1a_hour_*.parquet"), read_eea("utd_hour_*.parquet")])
    series = {}
    for pcode, name in [(7, "o3"), (8, "no2")]:
        h = hour[hour["Pollutant"] == pcode]
        series[name] = h.groupby("Start")["Value"].mean()
    # PM: E408 MB Vrbanski hourly (2021-2025), SI0059A hourly (2026, sampling points _101)
    for f, pcode, name in [("LPetravic_PM10.dsv", 5, "pm10"), ("LPetravic_PM2_5.dsv", 6001, "pm25")]:
        n = pd.read_csv(RAW / "arso_air" / f, sep=";", decimal=",")
        n = n[(n["INSTALACIJA"] == "E408") & (n["VELJAVNOST_1H"] == 1)]
        e408 = n.groupby(pd.to_datetime(n["DATUM_ZAJEMA"], format="%d.%m.%Y %H:%M"))["POV_1H"].mean()
        si = hour[hour["Pollutant"] == pcode].groupby("Start")["Value"].mean()
        series[name] = e408.combine_first(si[si.index > e408.index.max()])
    conc = pd.DataFrame(series)
    levels = pd.DataFrame({p: eaqi_level(conc[p], p) for p in EAQI_BANDS})
    hourly = levels.max(axis=1, skipna=True).where(levels.notna().any(axis=1))
    day = hourly.groupby(hourly.index.normalize())
    out = pd.DataFrame({
        "aqi_max": day.max(),
        "aqi_mean": day.mean().round(2),
        "aqi_hours_poor": day.apply(lambda s: (s >= 4).sum()).astype(float),
        "aqi_n_hours": day.count().astype(float),
    })
    out.loc[out["aqi_n_hours"] < AQI_MIN_HOURS, ["aqi_max", "aqi_mean", "aqi_hours_poor"]] = np.nan
    global aqi_hourly_levels
    aqi_hourly_levels = levels
    return to_daily(out)


aqi = build_aqi()
arso_air = arso_air.join(aqi)
display(aqi["aqi_max"].value_counts().sort_index().rename(lambda v: f"{int(v)} {EAQI_LABELS[int(v)]}").rename("days (worst hour)").to_frame())
print("Pollutant setting the worst level, share of hours with an index:")
worst = aqi_hourly_levels.dropna(how="all").idxmax(axis=1)
(worst.value_counts(normalize=True) * 100).round(1).rename("% of hours").to_frame()

In [ ]:
aq = aqi.dropna(subset=["aqi_max"])
colors_aqi = {1: "#50f0e6", 2: "#50ccaa", 3: "#f0e641", 4: "#ff5050", 5: "#960032", 6: "#7d2181"}  # EEA index colours
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.bar(aq.index, np.ones(len(aq)), width=1, color=[colors_aqi[int(v)] for v in aq["aqi_max"]])
ax.set_yticks([]); ax.set_title("European Air Quality Index, worst hour of each day (Maribor)")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors_aqi.values()]
ax.legend(handles, [f"{k} {v}" for k, v in EAQI_LABELS.items()], ncol=6, loc="lower left", bbox_to_anchor=(0, 1.08), fontsize=8)
study_span(ax); fig.tight_layout()

### 2.4 Respiratory illness

**ERVISS** (Slovenia, weekly, from 2022-W25 only). The main columns hold each ISO week's value on all seven days of that week; `_rt` columns hold the last week already published (week end + `ERVISS_PUBLICATION_LAG_D`); `_interp` columns interpolate between weeks (sensitivity only):
- `ili_rate`, `ari_rate`: primary-care ILI/ARI consultation rate per 100,000, all ages
- `pos_flu`, `pos_rsv`, `pos_sars`: sentinel positivity (%)
- `det_flu`, `det_rsv`, `det_sars`: non-sentinel detections

**PCR testing, Maribor** (daily, 2021-01-01 to 2024-02-29): positive results per day, `pcr_sars`, `pcr_flu_a`, `pcr_flu_b`, `pcr_rsv`. This covers the period before ERVISS starts.

In [ ]:
def load_erviss() -> pd.DataFrame:
    d = RAW / "erviss"
    ili = pd.read_csv(d / "ILIARIRates.csv").query("countryname == 'Slovenia' and age == 'total'")
    ili = ili.pivot_table(index="yearweek", columns="indicator", values="value", aggfunc="first")
    ili = ili.rename(columns={"ILIconsultationrate": "ili_rate", "ARIconsultationrate": "ari_rate"})
    short = {"Influenza": "flu", "RSV": "rsv", "SARS-CoV-2": "sars"}

    def pathogen_totals(f, indicator, prefix):
        x = pd.read_csv(d / f).query("countryname == 'Slovenia' and age == 'total' and indicator == @indicator")
        x = x[x["pathogentype"] == x["pathogen"]]  # pathogen total, not subtypes
        x = x.pivot_table(index="yearweek", columns="pathogen", values="value", aggfunc="first")
        return x.rename(columns=lambda p: f"{prefix}_{short[p]}")

    weekly = pd.concat([
        ili,
        pathogen_totals("sentinelTestsDetectionsPositivity.csv", "positivity", "pos"),
        pathogen_totals("nonSentinelTestsDetections.csv", "detections", "det"),
    ], axis=1)
    weekly.columns.name = None
    week_start = pd.to_datetime(pd.Series(weekly.index) + "-1", format="%G-W%V-%u")
    register("ERVISS (Slovenia)", pd.concat([week_start, week_start + pd.Timedelta(days=6)]), "exposure", "weekly", "ECDC", "erviss/*.csv")
    spec = dict(kind="period", values=weekly.reset_index(drop=True), ref_start=week_start,
                ref_end=week_start + pd.Timedelta(days=6), anchor=week_start + pd.Timedelta(days=3),
                lag_days=ERVISS_PUBLICATION_LAG_D, max_carry_days=RT_MAX_CARRY_D)
    LOWFREQ_INPUTS["erviss"] = spec
    return period_versions(**{k: v for k, v in spec.items() if k != "kind"})


def load_pcr() -> pd.DataFrame:
    f = next((RAW / "PCR testing").glob("*.xls"))
    p = pd.read_excel(f, engine="calamine")
    p.columns = [c.strip() for c in p.columns]
    p["date"] = pd.to_datetime(p["Datum sprejema"].astype(str).str.replace(" ", ""), format="%d.%m.%Y")
    register("PCR testing (Maribor)", p["date"], "exposure", "daily", "NLZOH, Maribor", "PCR testing/" + f.name)
    p = p.set_index("date").rename(columns={"SARS-CoV-2": "pcr_sars", "INFa": "pcr_flu_a", "INFb": "pcr_flu_b", "RSV": "pcr_rsv"})
    return to_daily(p[["pcr_sars", "pcr_flu_a", "pcr_flu_b", "pcr_rsv"]])


erviss = load_erviss()
pcr = load_pcr()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(erviss.index, erviss["ili_rate"], label="ILI (week value)", color="C3", lw=1)
axes[0].plot(erviss.index, erviss["ari_rate"], label="ARI (week value)", color="C0", lw=1)
axes[0].plot(erviss.index, erviss["ari_rate_rt"], label="ARI real time (_rt)", color="C0", lw=0.8, ls=":")
axes[0].set_title("ERVISS primary-care consultation rate per 100,000 (Slovenia)"); axes[0].legend(loc="upper left")
for c, col in [("pos_flu", "C3"), ("pos_rsv", "C2"), ("pos_sars", "C7")]:
    axes[1].plot(erviss.index, erviss[c], label=c.replace("pos_", ""), color=col, lw=1)
axes[1].set_title("ERVISS sentinel positivity (%)"); axes[1].legend(loc="upper left")
pw = pcr.resample("W").sum(min_count=7)
pw["pcr_flu"] = pw["pcr_flu_a"] + pw["pcr_flu_b"]
for c, col in [("pcr_flu", "C3"), ("pcr_rsv", "C2")]:
    axes[2].plot(pw.index, pw[c], label=c.replace("pcr_", ""), color=col, lw=1)
ax2 = axes[2].twinx(); ax2.plot(pw.index, pw["pcr_sars"], color="C7", lw=1, label="SARS-CoV-2 (right)")
ax2.spines["top"].set_visible(False); ax2.grid(False)
axes[2].set_title("PCR positives per week, Maribor"); axes[2].legend(loc="upper left"); ax2.legend(loc="upper right")
study_span(axes[2]); fig.tight_layout()

#### Combined influenza index (adjustment for H1 over the whole study period)

Influenza and cold season coincide, so the temperature model (H1) is adjusted for influenza activity. ERVISS starts only in mid-2022, so a **combined weekly index** is built:

- from 2022-W25: **ERVISS non-sentinel influenza detections** (Slovenia, lab-confirmed, per ISO week);
- before that: **Maribor PCR influenza positives** (A + B, NLZOH), summed per complete ISO week and **scaled** to the ERVISS level with the ratio of the two series over the weeks where both exist.

Both series count lab-confirmed influenza, and in the overlap they move together closely (correlation shown below; the run stops if it falls below 0.8). Before mid-2022 influenza was nearly absent, so the spliced part carries little weight. Each day takes its ISO week's value (no look-ahead within the explanatory use); `flu_index_source` marks the source. There is no real-time version; for forecasting use `det_flu_rt` and lagged PCR.

In [ ]:
FLU_MIN_OVERLAP_R = 0.8


def build_flu_index() -> pd.DataFrame:
    spec = LOWFREQ_INPUTS["erviss"]
    week_start = pd.DatetimeIndex(spec["ref_start"])
    erviss_week = pd.Series(spec["values"]["det_flu"].to_numpy(dtype=float), index=week_start).dropna()
    # Maribor PCR influenza positives per complete ISO week (Monday start)
    daily_flu = pcr["pcr_flu_a"] + pcr["pcr_flu_b"]
    weeks = daily_flu.groupby(daily_flu.index.to_period("W-SUN").start_time)
    pcr_week = weeks.sum(min_count=7).where(weeks.count() == 7).dropna()

    overlap = pd.concat([pcr_week.rename("pcr"), erviss_week.rename("erviss")], axis=1).dropna()
    r = np.corrcoef(np.log1p(overlap["pcr"]), np.log1p(overlap["erviss"]))[0, 1]
    scale = overlap["erviss"].sum() / overlap["pcr"].sum()
    print(f"Overlap: {len(overlap)} weeks ({overlap.index.min():%Y-%m-%d} to {overlap.index.max():%Y-%m-%d}); "
          f"correlation (log scale) r = {r:.3f}; scale factor ERVISS / Maribor PCR = {scale:.2f}")
    assert r >= FLU_MIN_OVERLAP_R, f"PCR and ERVISS influenza disagree in the overlap (r = {r:.2f}): do not splice"

    before = pcr_week[pcr_week.index < erviss_week.index.min()] * scale
    weekly = pd.concat([before.to_frame("flu_index").assign(flu_index_source=0),
                        erviss_week.to_frame("flu_index").assign(flu_index_source=1)]).sort_index()
    assert weekly.index.is_unique
    register("Influenza index (PCR + ERVISS)", [weekly.index.min(), weekly.index.max() + pd.Timedelta(days=6)], "exposure", "weekly",
             "NLZOH + ECDC (derived)", "PCR testing/*.xls, erviss/nonSentinelTestsDetections.csv")
    daily = weekly.reindex(pd.date_range(weekly.index.min(), weekly.index.max() + pd.Timedelta(days=6), freq="D")).ffill(limit=6)
    global flu_overlap
    flu_overlap = overlap.assign(pcr_scaled=overlap["pcr"] * scale)
    return to_daily(daily)


flu_index = build_flu_index()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 2.4]})
ax = axes[0]
ax.scatter(np.log1p(flu_overlap["pcr_scaled"]), np.log1p(flu_overlap["erviss"]), s=10, color="C3", alpha=0.6)
lim = [0, np.log1p(flu_overlap[["pcr_scaled", "erviss"]].max().max()) + 0.3]
ax.plot(lim, lim, color="0.3", lw=0.8, ls="--")
ax.set_xlabel("log(1 + Maribor PCR, scaled)"); ax.set_ylabel("log(1 + ERVISS detections)")
ax.set_title("Overlap weeks: PCR vs ERVISS")
ax = axes[1]
f = flu_index
ax.fill_between(f.index, 0, f["flu_index"].where(f["flu_index_source"] == 0), step="post", color="C1", alpha=0.6, label="Maribor PCR (scaled)")
ax.fill_between(f.index, 0, f["flu_index"].where(f["flu_index_source"] == 1), step="post", color="C3", alpha=0.6, label="ERVISS detections")
ax.set_title("Combined influenza index (detections per week, ERVISS scale)"); ax.legend(loc="upper left")
study_span(ax); fig.tight_layout()

### 2.5 COVID-19, Štajerska (Sledilnik)

From the Sledilnik API (`https://api.sledilnik.org/api/`), cached as JSON in `data/raw/covid_sledilnik/`. Files are downloaded only if missing; set `COVID_FORCE_DOWNLOAD = True` to refresh. Sledilnik stopped updating in spring 2023.

**Štajerska** = Podravska (`mb`) + Savinjska (`ce`), the same definition as for NDVI.

| Column | Meaning | Endpoint |
|---|---|---|
| `covid_cases_mb`, `covid_cases_ce`, `covid_cases_stajerska` | New confirmed cases per day (difference of the cumulative count) | `regions` |
| `covid_active_stajerska` | Active cases | `regions` |
| `covid_deaths_stajerska` | New deaths per day | `regions` |
| `covid_ukcmb_inpatients`, `covid_ukcmb_icu` | COVID-19 patients in UKC Maribor (ward, ICU) | `patients` |
| `covid_ukcmb_admissions` | New COVID-19 admissions to UKC Maribor | `patients` |
| `covid_sewage_mb` | SARS-CoV-2 N1 copies normalised to PMMoV, Maribor wastewater plant: each sample carried forward from its sampling date for up to `SEWAGE_MAX_CARRY_D` days. Also `_rt` (from the publication date), `_interp` (sensitivity) and `_age_days` (days since the sample). | `sewage` |

Case counts are by reporting date and drop on weekends; use a 7-day mean or a day-of-week adjustment.

In [ ]:
COVID_DIR = RAW / "covid_sledilnik"
COVID_API = "https://api.sledilnik.org/api/"
COVID_ENDPOINTS = ["regions", "patients", "sewage", "hospitals", "hospitals-list", "municipalities"]
COVID_FORCE_DOWNLOAD = False
STAJERSKA_REGIONS = {"mb": "Podravska", "ce": "Savinjska"}

COVID_DIR.mkdir(parents=True, exist_ok=True)
for ep in COVID_ENDPOINTS:
    f = COVID_DIR / f"{ep}.json"
    if COVID_FORCE_DOWNLOAD or not f.exists():
        r = requests.get(COVID_API + ep, timeout=300)
        r.raise_for_status()
        f.write_bytes(r.content)
        print(f"downloaded {ep}: {len(r.content) / 1e6:.1f} MB")


def sledilnik_date(d):
    return pd.Timestamp(year=d["year"], month=d["month"], day=d["day"])


def load_covid() -> pd.DataFrame:
    regions = json.loads((COVID_DIR / "regions.json").read_text())
    rows = []
    for d in regions:
        row = {"date": sledilnik_date(d)}
        for code in STAJERSKA_REGIONS:
            r = d["regions"].get(code, {})
            row[f"confirmed_{code}"] = r.get("confirmedToDate")
            row[f"active_{code}"] = r.get("activeCases")
            row[f"deceased_{code}"] = r.get("deceasedToDate")
        rows.append(row)
    reg = pd.DataFrame(rows).set_index("date").sort_index()
    register("COVID-19 cases Štajerska (Sledilnik)", reg.index, "exposure", "daily", "Sledilnik", "covid_sledilnik/regions.json")

    out = pd.DataFrame(index=reg.index)
    for code in STAJERSKA_REGIONS:
        out[f"covid_cases_{code}"] = reg[f"confirmed_{code}"].diff().clip(lower=0)
    out["covid_cases_stajerska"] = out[[f"covid_cases_{c}" for c in STAJERSKA_REGIONS]].sum(axis=1, min_count=1)
    out["covid_active_stajerska"] = reg[[f"active_{c}" for c in STAJERSKA_REGIONS]].sum(axis=1, min_count=1)
    out["covid_deaths_stajerska"] = reg[[f"deceased_{c}" for c in STAJERSKA_REGIONS]].sum(axis=1, min_count=1).diff().clip(lower=0)

    patients = json.loads((COVID_DIR / "patients.json").read_text())
    p = pd.DataFrame([{
        "date": sledilnik_date(d),
        "covid_ukcmb_inpatients": d.get("facilities", {}).get("ukcmb", {}).get("inHospital", {}).get("today"),
        "covid_ukcmb_icu": d.get("facilities", {}).get("ukcmb", {}).get("icu", {}).get("today"),
        "covid_ukcmb_admissions": d.get("facilities", {}).get("ukcmb", {}).get("inHospital", {}).get("in"),
    } for d in patients]).set_index("date").sort_index()
    register("COVID-19 patients UKC Maribor (Sledilnik)", p.dropna(how="all").index, "exposure", "daily", "Sledilnik", "covid_sledilnik/patients.json")

    sewage = json.loads((COVID_DIR / "sewage.json").read_text())
    s = pd.Series({sledilnik_date(d): d["plants"]["maribor"].get("cp-luc-pmmov-rawpmmov-n1")
                   for d in sewage if "maribor" in d.get("plants", {})}, dtype=float).dropna().sort_index()
    register("COVID-19 wastewater Maribor (Sledilnik)", s.index, "exploratory", "~weekly samples", "Sledilnik / NIB", "covid_sledilnik/sewage.json")
    global covid_sewage_samples
    covid_sewage_samples = s
    sew = s.to_frame("covid_sewage_mb")
    LOWFREQ_INPUTS["covid_sewage_mb"] = dict(kind="sample", values=sew, lag_days=SEWAGE_LAB_LAG_D, max_carry_days=SEWAGE_MAX_CARRY_D)
    sew_daily = sample_versions(sew, SEWAGE_LAB_LAG_D, SEWAGE_MAX_CARRY_D)

    covid = to_daily(out.join(p, how="outer"))
    return covid.join(sew_daily)


covid = load_covid()

In [ ]:
c = covid.dropna(how="all")
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for col, color in [("covid_cases_mb", "C3"), ("covid_cases_ce", "C0")]:
    axes[0].plot(c.index, c[col].rolling(7, center=True).mean(), color=color, lw=1.1, label=f"{STAJERSKA_REGIONS[col[-2:]]} (7-day mean)")
axes[0].set_title("New confirmed COVID-19 cases per day, Štajerska regions"); axes[0].legend(loc="upper left")
axes[1].plot(c.index, c["covid_ukcmb_inpatients"], color="C5", lw=1, label="inpatients")
axes[1].plot(c.index, c["covid_ukcmb_icu"], color="C3", lw=1, label="ICU")
axes[1].bar(c.index, c["covid_ukcmb_admissions"], width=1, color="0.6", label="admissions")
axes[1].set_title("COVID-19 patients in UKC Maribor"); axes[1].legend(loc="upper left")
axes[2].plot(c.index, c["covid_sewage_mb"], color="C2", lw=1, drawstyle="steps-post", label="carried forward (main)")
axes[2].plot(c.index, c["covid_sewage_mb_interp"], color="C2", lw=0.8, ls=":", label="interpolated (sensitivity)")
axes[2].scatter(covid_sewage_samples.index, covid_sewage_samples.values, s=8, color="C2", label="samples")
axes[2].set_yscale("log"); axes[2].set_title("SARS-CoV-2 in Maribor wastewater (N1 / PMMoV)"); axes[2].legend(loc="upper left")
study_span(axes[2]); axes[2].set_xlim(START, END); fig.tight_layout()

### 2.6 Pollen (Maribor, 2021–2023, exploratory)

Daily pollen counts for 13 taxa plus the reported total (`pollen_total`). The sheet has one row per day of year and, for each taxon, one column per year.

In [ ]:
def load_pollen() -> pd.DataFrame:
    p = pd.read_excel(RAW / "pollen" / "pollen.xlsx", header=None, engine="calamine")
    body = p.iloc[6:].copy()
    doy = pd.to_datetime(body[0])
    out = {}
    # Each block = a name in row 0 (Slovene) / row 1 (Latin), then one column per year (row 5).
    # The last block, 'skupno število', is the reported total over all taxa.
    blocks = [c for c in range(1, p.shape[1]) if pd.notna(p.iat[0, c])]
    for b in blocks:
        latin = p.iat[1, b]
        name = "pollen_" + (snake(str(latin).split("/")[0]) if pd.notna(latin) else "total")
        for col in range(b, b + 3):
            year = p.iat[5, col]
            dates = pd.to_datetime({"year": int(year), "month": doy.dt.month, "day": doy.dt.day})
            out.setdefault(name, []).append(pd.Series(pd.to_numeric(body[col], errors="coerce").values, index=dates))
    pol = pd.DataFrame({k: pd.concat(v).sort_index() for k, v in out.items()})
    assert pol.index.is_unique and "pollen_total" in pol
    register("Pollen (Maribor)", pol.index, "exploratory", "daily", "Pollen station Maribor", "pollen/pollen.xlsx")
    return to_daily(pol)


pollen = load_pollen()

In [ ]:
taxa = [c for c in pollen if c != "pollen_total"]
wk = pollen.loc[: "2023-12-31", taxa].resample("W").sum(min_count=7)
fig, ax = plt.subplots(figsize=(12, 4.5))
im = ax.imshow(np.log10(wk.T.values + 1), aspect="auto", cmap="YlOrBr", interpolation="nearest",
               extent=[mdates.date2num(wk.index[0]), mdates.date2num(wk.index[-1]), len(taxa) - 0.5, -0.5])
ax.set_yticks(range(len(taxa)), [t.replace("pollen_", "") for t in taxa])
ax.xaxis_date(); ax.grid(False)
fig.colorbar(im, ax=ax, label="log10(weekly count + 1)", pad=0.01)
ax.set_title("Pollen by taxon, weekly sum (Maribor)")
fig.tight_layout()

### 2.7 Google Trends

Monthly index (0–100) for *"urgenca maribor"*. The main column holds each month's value on every day of that month (a monthly exposure: it cannot support daily lags). `_rt` holds the last complete month (available the day after the month ends); `_interp` interpolates between mid-months (sensitivity only). Weekly or daily Trends would be needed for daily lag analyses.

In [ ]:
def load_google_trends() -> pd.DataFrame:
    out = []
    for f in sorted((RAW / "google_trends").glob("*.csv")):
        g = pd.read_csv(f)
        g["Time"] = pd.to_datetime(g["Time"])
        g = g.set_index("Time")
        g = g.apply(lambda s: pd.to_numeric(s.astype(str).str.replace("<1", "0.5"), errors="coerce"))
        g.columns = ["gt_" + snake(c) for c in g.columns]
        out.append(g)
    g = pd.concat(out, axis=1)
    register("Google Trends", [g.index.min(), g.index.max() + pd.offsets.MonthEnd(0)], "exposure", "monthly", "Google", "google_trends/*.csv")
    global google_trends_monthly
    google_trends_monthly = g
    month_start = pd.Series(g.index)
    spec = dict(kind="period", values=g.reset_index(drop=True), ref_start=month_start,
                ref_end=month_start + pd.offsets.MonthEnd(0), anchor=month_start + pd.Timedelta(days=14),
                lag_days=GT_PUBLICATION_LAG_D, max_carry_days=GT_MAX_CARRY_D)
    LOWFREQ_INPUTS["google_trends"] = spec
    return period_versions(**{k: v for k, v in spec.items() if k != "kind"})


google_trends = load_google_trends()

In [ ]:
fig, ax = plt.subplots()
for c in google_trends_monthly:
    ax.step(google_trends_monthly.index, google_trends_monthly[c], where="post", lw=1, label=c)
ax.set_title("Google Trends, monthly index (full download)"); ax.legend(loc="upper left")
study_span(ax); fig.tight_layout()

### 2.8 SBI TOP

Daily close and log return on trading days. Non-trading days are `NaN`, and `trading_day` flags them. For forecasting, `sbi_close_rt` carries the last close over non-trading days and `sbi_return_rt` is then 0 on those days.

In [ ]:
def load_sbi_top() -> pd.DataFrame:
    s = pd.read_csv(RAW / "sbi_top" / "SBI.csv", sep=";", decimal=",", parse_dates=["date"])
    register("SBI TOP", s["date"], "exposure", "trading days", "Ljubljana Stock Exchange", "sbi_top/SBI.csv")
    s = s.set_index("date").sort_index()
    out = pd.DataFrame({"sbi_close": s["last_value"], "sbi_return": np.log(s["last_value"]).diff()})
    out = to_daily(out)
    out["trading_day"] = out["sbi_close"].notna().astype(int)
    out["sbi_close_rt"] = out["sbi_close"].ffill(limit=7)
    out["sbi_return_rt"] = np.log(out["sbi_close_rt"]).diff()
    return out


sbi_top = load_sbi_top()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(sbi_top.index, sbi_top["sbi_close"], color="C0", lw=1)
axes[0].set_title("SBI TOP close")
r = sbi_top["sbi_return"].dropna()
axes[1].bar(r.index, r * 100, width=1, color=np.where(r < 0, "C3", "C2"))
axes[1].set_title("Daily log return (%)")
study_span(axes[1]); fig.tight_layout()

### 2.9 NDVI (Sentinel-2; preregistered, treated as exploratory)

Sentinel-2 L2A scenes from Microsoft Planetary Computer (no account needed), for two areas:
- `maribor`: bounding box around the city (city, Pohorje foothills, edge of the Drava plain), at 20 m
- `stajerska`: NUTS-3 regions Podravska (SI032) + Savinjska (SI034), polygon from Eurostat GISCO, at 100 m

Per acquisition date: mean NDVI over clear pixels (SCL classes 4–7). Scenes more than `NDVI_OUTLIER_DROP` below both neighbouring scenes (snow, undetected cloud) are dropped. Daily columns follow the temporal-alignment rules above: main = last scene carried forward (max `NDVI_MAX_CARRY_D` days), plus `_rt`, `_interp` and `_age_days`. A date is kept only if ≥ 50% of the area is clear. From processing baseline 04.00 (25 Jan 2022) on, the +1000 reflectance offset is removed. Results are cached in `data/raw/ndvi/<area>_scenes.csv`; the download only runs if a file is missing, or if `NDVI_FORCE_DOWNLOAD = True`. A full download takes a while.

In [ ]:
NDVI_DIR = RAW / "ndvi"
NDVI_START = (START - pd.DateOffset(months=6)).strftime("%Y-%m-%d")
NDVI_END = END.strftime("%Y-%m-%d")
NDVI_FORCE_DOWNLOAD = False
MAX_SCENE_CLOUD = 60      # % cloud of the whole tile (coarse prefilter)
MIN_VALID_FRAC = 0.5      # keep a date only if >= 50% of area pixels are clear
CLEAR_SCL = [4, 5, 6, 7]  # SCL: vegetation, bare soil, water, unclassified

STAJERSKA_NUTS = ["SI032", "SI034"]   # Podravska + Savinjska
GISCO_URL = ("https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/"
             "NUTS_RG_01M_2021_4326_LEVL_3.geojson")
MARIBOR_BBOX = (15.55, 46.50, 15.75, 46.62)
UKC_MARIBOR = (15.6447, 46.5541)  # lon, lat


def slovenia_nuts3() -> dict:
    """NUTS-3 polygons of Slovenia from GISCO, cached in data/raw/ndvi/."""
    cache = NDVI_DIR / "nuts3_si.geojson"
    if not cache.exists():
        fc = requests.get(GISCO_URL, timeout=120).json()
        fc["features"] = [f for f in fc["features"] if f["properties"]["CNTR_CODE"] == "SI"]
        cache.write_text(json.dumps(fc))
    fc = json.loads(cache.read_text())
    return {f["properties"]["NUTS_ID"]: (f["properties"]["NUTS_NAME"], shape(f["geometry"])) for f in fc["features"]}


try:
    NUTS3 = slovenia_nuts3()
except Exception as e:  # noqa: BLE001
    raise RuntimeError(f"NUTS-3 borders could not be loaded ({type(e).__name__}): Štajerska cannot be defined. "
                       "Check the connection to GISCO; the area is never replaced by a bounding box.") from e
STAJERSKA = unary_union([NUTS3[c][1] for c in STAJERSKA_NUTS])

# name -> (geometry in WGS84, pixel size in metres)
NDVI_AREAS = {
    "maribor": (box(*MARIBOR_BBOX), 20),
    "stajerska": (STAJERSKA, 100),
}

In [ ]:
def draw(ax, geom, **kw):
    for g in getattr(geom, "geoms", [geom]):
        x, y = g.exterior.xy
        ax.fill(x, y, **kw)


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [1.6, 1]})
ax = axes[0]
for code_, (name, g) in NUTS3.items():
    sel = code_ in STAJERSKA_NUTS
    draw(ax, g, facecolor="#a5d6a7" if sel else "#f2f2f2", edgecolor="0.4", lw=0.6)
    c = g.representative_point()
    ax.text(c.x, c.y, name, fontsize=7, ha="center", color="0.15" if sel else "0.45")
draw(ax, STAJERSKA, facecolor="none", edgecolor="#1565c0", lw=2)
draw(ax, NDVI_AREAS["maribor"][0], facecolor="none", edgecolor="#c62828", lw=2)
ax.plot(*UKC_MARIBOR, marker="*", color="#c62828", ms=12)
ax.set_title("NDVI areas: Štajerska = Podravska + Savinjska (blue), Maribor box (red)")
ax.set_aspect(1 / np.cos(np.radians(46.1))); ax.grid(False)
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")

ax = axes[1]
for code_, (name, g) in NUTS3.items():
    draw(ax, g, facecolor="#a5d6a7" if code_ in STAJERSKA_NUTS else "#f2f2f2", edgecolor="0.4", lw=0.6)
draw(ax, NDVI_AREAS["maribor"][0], facecolor="#c62828", alpha=0.12, edgecolor="#c62828", lw=2)
ax.plot(*UKC_MARIBOR, marker="*", color="#c62828", ms=14)
ax.annotate("UKC Maribor", UKC_MARIBOR, xytext=(6, 6), textcoords="offset points", fontsize=8)
pad = 0.05
ax.set_xlim(MARIBOR_BBOX[0] - pad, MARIBOR_BBOX[2] + pad); ax.set_ylim(MARIBOR_BBOX[1] - pad, MARIBOR_BBOX[3] + pad)
ax.set_aspect(1 / np.cos(np.radians(46.56))); ax.grid(alpha=0.3)
ax.set_title(f"Maribor box {MARIBOR_BBOX}")
fig.tight_layout()

In [ ]:
def ndvi_series(name, geom, resolution):
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)
    items = list(catalog.search(
        collections=["sentinel-2-l2a"], intersects=mapping(geom),
        datetime=f"{NDVI_START}/{NDVI_END}", query={"eo:cloud_cover": {"lt": MAX_SCENE_CLOUD}},
    ).items())
    print(f"[{name}] {len(items)} Sentinel-2 items")
    aoi = Geometry(mapping(geom), crs="EPSG:4326")
    ds = stac_load(items, bands=["B04", "B08", "SCL"], geopolygon=aoi, crs="EPSG:32633", resolution=resolution,
              groupby="solar_day", chunks={"x": 1024, "y": 1024})
    inside = rasterize(aoi, ds.odc.geobox)  # only pixels inside the polygon count

    # Remove the +1000 DN offset from processing baseline 04.00 (25 Jan 2022)
    offset = (ds.time >= np.datetime64("2022-01-25")).astype("int16") * 1000
    red = (ds.B04.astype("float32") - offset).clip(min=0) / 10000
    nir = (ds.B08.astype("float32") - offset).clip(min=0) / 10000
    clear = ds.SCL.isin(CLEAR_SCL) & inside
    ndvi = ((nir - red) / (nir + red)).where(clear & (nir + red > 0))

    n_inside = int(inside.sum())
    print(f"[{name}] computing over {ds.sizes['time']} dates ({n_inside:,} pixels at {resolution} m)...")
    scenes = pd.DataFrame({
        "date": pd.to_datetime(ds.time.values).normalize(),
        "ndvi_mean": ndvi.mean(dim=["x", "y"]).compute().values,
        "valid_frac": (clear.sum(dim=["x", "y"]) / n_inside).compute().values,
    })
    scenes = scenes[scenes.valid_frac >= MIN_VALID_FRAC].dropna()
    scenes.to_csv(NDVI_DIR / f"{name}_scenes.csv", index=False)
    print(f"[{name}] kept {len(scenes)} clear dates")
    return scenes


for name, (geom, res) in NDVI_AREAS.items():
    if NDVI_FORCE_DOWNLOAD or not (NDVI_DIR / f"{name}_scenes.csv").exists():
        ndvi_series(name, geom, res)

In [ ]:
def load_ndvi() -> pd.DataFrame | None:
    out, scenes = {}, {}
    for area in NDVI_AREAS:
        f = NDVI_DIR / f"{area}_scenes.csv"
        if not f.exists():
            continue
        s = pd.read_csv(f, parse_dates=["date"]).groupby("date")["ndvi_mean"].mean().sort_index()
        register(f"NDVI {area}", s.index, "exploratory", "per clear scene", "Copernicus Sentinel-2", f"ndvi/{f.name}")
        # Snow / undetected cloud: a scene far below both neighbouring scenes
        outlier = (s < s.shift(1) - NDVI_OUTLIER_DROP) & (s < s.shift(-1) - NDVI_OUTLIER_DROP)
        print(f"NDVI {area}: {len(s)} scenes, {int(outlier.sum())} outliers dropped {s.index[outlier].strftime('%Y-%m-%d').tolist()}")
        s = s[~outlier]
        scenes[area] = s
        values = s.to_frame(f"ndvi_{area}")
        LOWFREQ_INPUTS[f"ndvi_{area}"] = dict(kind="sample", values=values, lag_days=NDVI_LAG_D, max_carry_days=NDVI_MAX_CARRY_D)
        out[area] = sample_versions(values, NDVI_LAG_D, NDVI_MAX_CARRY_D)
    if not out:
        print("NDVI: no scenes yet, skipped")
        return None, scenes
    return pd.concat(out.values(), axis=1), scenes


ndvi, ndvi_scenes = load_ndvi()
if ndvi is not None:
    fig, ax = plt.subplots()
    for (area, s), col in zip(ndvi_scenes.items(), ["#2e7d32", "#1565c0"]):
        ax.scatter(s.index, s.values, s=6, alpha=0.35, color=col)
        ax.plot(ndvi.index, ndvi[f"ndvi_{area}"], color=col, lw=1, drawstyle="steps-post", label=f"{area} (carried forward)")
    ax.set_ylim(0, 1); ax.set_title("Sentinel-2 NDVI (dots = clear scenes)"); ax.legend(loc="lower left")
    study_span(ax); fig.tight_layout()

### 2.10 EMS dispatch, Maribor (exploratory)

Dispatch cards of the Maribor EMS, 2024-08-30 to 2026-10-13. Not in the preregistration.

- `ems_calls`: calls per day
- `ems_vuz`: calls with `sni_vuz` = DA

The first weeks are a roll-out with only a few calls a day, so they are dropped.

In [ ]:
def load_ems() -> pd.DataFrame:
    e = pd.read_excel(RAW / "EMS" / "Maribor.xlsx", sheet_name="sni_edited", engine="calamine")
    e["date"] = pd.to_datetime(e["sni_datum_dogodka"])
    g = e.groupby("date")
    out = pd.DataFrame({"ems_calls": g.size(), "ems_vuz": g["sni_vuz"].apply(lambda s: (s == "DA").sum())})
    full = pd.date_range(out.index.min(), out.index.max(), freq="D")
    out = out.reindex(full, fill_value=0)
    # Roll-out: the first weeks hold only a fraction of the usual calls. Start at the first day
    # whose 7-day mean reaches half of the overall median.
    steady = out["ems_calls"].rolling(7).mean() >= 0.5 * out["ems_calls"].median()
    first = steady.idxmax() - pd.Timedelta(days=6)
    print(f"EMS: roll-out period {out.index.min():%Y-%m-%d} – {first - pd.Timedelta(days=1):%Y-%m-%d} dropped")
    register("EMS dispatch (Maribor)", out.loc[first:].index, "exploratory", "per call", "EMS Maribor", "EMS/Maribor.xlsx")
    return to_daily(out.loc[first:])


ems = load_ems()

In [ ]:
e = ems.dropna(how="all")
fig, ax = plt.subplots()
ax.plot(e.index, e["ems_calls"], color="C0", lw=0.5, alpha=0.4)
ax.plot(e.index, e["ems_calls"].rolling(7, center=True).mean(), color="C0", lw=1.2, label="calls (7-day mean)")
ax.plot(e.index, e["ems_vuz"].rolling(7, center=True).mean(), color="C3", lw=1.2, label="vuz = DA (7-day mean)")
ax.set_title("EMS dispatch calls per day, Maribor"); ax.legend(loc="upper left")
fig.tight_layout()

### 2.11 Football fixtures, MNZ Maribor (exploratory)

Amateur and youth football matches in the Maribor regional football association (MNZ Maribor), 2021-04 to 2024-05. `football_matches` is the number of matches per day. A possible driver of sports injuries; not in the preregistration.

In [ ]:
def load_football() -> pd.DataFrame:
    f = next((RAW / "NZS").glob("*.csv"))
    n = pd.read_csv(f, sep="\t", encoding="utf-16")
    n["date"] = pd.to_datetime(n["Datum"], format="%d.%m.%Y")
    register("Football fixtures (MNZ Maribor)", n["date"], "exploratory", "per match", "NZS", "NZS/" + f.name)
    s = n.groupby("date").size().rename("football_matches")
    full = pd.date_range(s.index.min(), s.index.max(), freq="D")
    return to_daily(s.reindex(full, fill_value=0).to_frame())


football = load_football()

In [ ]:
fw = football["football_matches"].resample("W").sum(min_count=7)
fig, ax = plt.subplots()
ax.bar(fw.index, fw, width=6, color="C2")
ax.set_title("Football matches per week (MNZ Maribor)")
study_span(ax); fig.tight_layout()

### 2.12 Hospital occupancy, UKC Maribor (exploratory)

Hourly occupancy by department (55 departments), 2021-01 to 2024-10. `hosp_occupancy` is the daily mean, over the hours, of total occupancy across all departments.

Occupancy partly results from ED arrivals (reverse causality), so use it only lagged, or as a descriptor. Not in the preregistration.

In [ ]:
def load_hospital_capacity() -> pd.DataFrame:
    c = pd.read_csv(RAW / "hospital capacity" / "Zasedenost oddelkov3.csv", sep=";", encoding="utf-8-sig",
                    usecols=["datum", "casod", "st"], parse_dates=["datum"])
    register("Hospital occupancy", c["datum"], "exploratory", "hourly", "UKC Maribor", "hospital capacity/Zasedenost oddelkov3.csv")
    hourly = c.groupby(["datum", "casod"])["st"].sum()
    return to_daily(hourly.groupby("datum").mean().rename("hosp_occupancy").to_frame())


hospital_capacity = load_hospital_capacity()

In [ ]:
h = hospital_capacity["hosp_occupancy"]
fig, ax = plt.subplots()
ax.plot(h.index, h, color="C5", lw=0.6, alpha=0.5, label="daily")
ax.plot(h.index, h.rolling(28, center=True).mean(), color="C5", lw=1.4, label="28-day mean")
ax.set_title("Hospital occupancy: daily mean of hourly total across departments"); ax.legend(loc="lower right")
study_span(ax); fig.tight_layout()

### 2.13 Moon phase (exploratory)

Illumination and moon age are **calculated** (Meeus, *Astronomical Algorithms*, ch. 48, low-precision series). The days of the full and new moon come from the official phase times of the **US Naval Observatory** (`aa.usno.navy.mil/api/moon/phases`, downloaded once and cached in `data/raw/moon/`), converted to local time (Europe/Ljubljana).

The calculation is checked against USNO: every new moon, first quarter, full moon and last quarter must be within 60 minutes. It agrees to within ~25 minutes, but when an event falls close to midnight the calculated local day can differ by one; those cases are listed below, which is why the event days use USNO.

| Column | Meaning |
|---|---|
| `moon_illumination` | Illuminated fraction of the disc at local noon (0–1) |
| `moon_age_days` | Days since the last new moon, at local noon (0–29.5) |
| `moon_phase` | 8 categories: 0 new, 1 waxing crescent, 2 first quarter, 3 waxing gibbous, 4 full, 5 waning gibbous, 6 last quarter, 7 waning crescent |
| `full_moon`, `new_moon` | 1 on the local day of the full / new moon |
| `full_moon_window` | Full moon day ± 1 day (common definition in "full-moon effect" studies) |

In [ ]:
SYNODIC_MONTH = 29.530588853
USNO_API = "https://aa.usno.navy.mil/api/moon/phases/year?year={year}"
MOON_DIR = RAW / "moon"
MOON_DIR.mkdir(parents=True, exist_ok=True)


def moon_phase(times_utc: pd.DatetimeIndex) -> tuple[np.ndarray, np.ndarray]:
    """Illuminated fraction and phase angle in degrees (0 = new, 90 = first quarter, 180 = full,
    270 = last quarter). Meeus, Astronomical Algorithms, ch. 48 (low-precision series)."""
    T = (times_utc.to_julian_date().to_numpy() - 2451545.0) / 36525
    D = np.radians((297.8501921 + 445267.1114034 * T) % 360)    # mean elongation of the Moon
    M = np.radians((357.5291092 + 35999.0502909 * T) % 360)     # Sun's mean anomaly
    Mp = np.radians((134.9633964 + 477198.8675055 * T) % 360)   # Moon's mean anomaly
    i = (180 - np.degrees(D) - 6.289 * np.sin(Mp) + 2.100 * np.sin(M) - 1.274 * np.sin(2 * D - Mp)
         - 0.658 * np.sin(2 * D) - 0.214 * np.sin(2 * Mp) - 0.110 * np.sin(D))  # phase angle
    return (1 + np.cos(np.radians(i))) / 2, (180 - i) % 360


def phase_events(start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    t = pd.date_range(start - pd.Timedelta(days=2), end + pd.Timedelta(days=2), freq="min", tz="UTC")
    _, ph = moon_phase(t)
    prev, cur = ph[:-1], ph[1:]
    events = []
    for name, angle in [("New Moon", 0), ("First Quarter", 90), ("Full Moon", 180), ("Last Quarter", 270)]:
        if angle == 0:
            hit = (prev > 270) & (cur < 90)                # wraps from 360 to 0
        else:
            hit = (prev < angle) & (cur >= angle)
        events += [(name, x) for x in t[1:][hit]]
    ev = pd.DataFrame(events, columns=["phase", "time_utc"]).sort_values("time_utc")
    ev["date_local"] = ev["time_utc"].dt.tz_convert("Europe/Ljubljana").dt.tz_localize(None).dt.normalize()
    return ev.reset_index(drop=True)


def load_usno(years) -> pd.DataFrame:
    frames = []
    for y in years:
        f = MOON_DIR / f"usno_phases_{y}.json"
        if not f.exists():
            r = requests.get(USNO_API.format(year=y), timeout=60)
            r.raise_for_status()
            f.write_text(r.text)
        frames.append(pd.DataFrame(json.loads(f.read_text())["phasedata"]))
    u = pd.concat(frames, ignore_index=True)
    u["time_utc"] = pd.to_datetime(u[["year", "month", "day"]]).dt.tz_localize("UTC") + pd.to_timedelta(u["time"] + ":00")
    u["date_local"] = u["time_utc"].dt.tz_convert("Europe/Ljubljana").dt.tz_localize(None).dt.normalize()
    return u[["phase", "time_utc", "date_local"]]


def build_moon() -> pd.DataFrame:
    noon_utc = (DAYS + pd.Timedelta(hours=12)).tz_localize("Europe/Ljubljana").tz_convert("UTC")
    illum, ph = moon_phase(noon_utc)
    moon = pd.DataFrame({
        "moon_illumination": illum.round(4),
        "moon_age_days": (ph / 360 * SYNODIC_MONTH).round(2),
        "moon_phase": (((ph + 22.5) % 360) // 45).astype(int),
    }, index=DAYS)
    global moon_events, usno
    moon_events = phase_events(START, END)
    usno = load_usno(range(START.year, END.year + 1))
    for col, name in [("full_moon", "Full Moon"), ("new_moon", "New Moon")]:
        moon[col] = DAYS.isin(usno.loc[usno["phase"] == name, "date_local"]).astype(int)
    moon["full_moon_window"] = moon["full_moon"].rolling(3, center=True, min_periods=1).max().astype(int)
    register("Moon phase (calculated + USNO)", [START, END], "exploratory", "daily", "Calculated (Meeus) + US Naval Observatory", "moon/usno_phases_*.json")
    return moon


moon = build_moon()

# Check the calculation against the US Naval Observatory
off = usno[(usno["date_local"] >= START) & (usno["date_local"] <= END)]
calc = moon_events[(moon_events["date_local"] >= START) & (moon_events["date_local"] <= END)]
cmp = pd.merge_asof(off.sort_values("time_utc"), calc.sort_values("time_utc").rename(columns={"time_utc": "calc_utc", "date_local": "calc_date"}),
                    left_on="time_utc", right_on="calc_utc", by="phase", direction="nearest")
cmp["diff_min"] = (cmp["calc_utc"] - cmp["time_utc"]).dt.total_seconds() / 60
cmp["same_day"] = cmp["calc_date"] == cmp["date_local"]
summary = cmp.groupby("phase").agg(events=("same_day", "size"), same_local_day=("same_day", "sum"),
                                   mean_abs_diff_min=("diff_min", lambda x: x.abs().mean()), max_abs_diff_min=("diff_min", lambda x: x.abs().max()))
assert abs(len(calc) - len(off)) <= 1, (len(calc), len(off))
assert cmp["diff_min"].abs().max() <= 60, cmp.loc[cmp["diff_min"].abs().idxmax()]
print("Events where the calculated local day differs from USNO (event close to midnight):")
display(cmp.loc[~cmp["same_day"], ["phase", "time_utc", "date_local", "calc_date", "diff_min"]])
summary.round(1)

In [ ]:
yr = moon.loc["2024"]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(yr.index, yr["moon_illumination"], color="0.4", lw=1, label="illumination at local noon")
ax.scatter(yr.index[yr["full_moon"] == 1], np.ones(int(yr["full_moon"].sum())), color="#f9a825", s=40, zorder=3, label="full moon")
ax.scatter(yr.index[yr["new_moon"] == 1], np.zeros(int(yr["new_moon"].sum())), color="black", s=40, zorder=3, label="new moon")
win = yr.index[yr["full_moon_window"] == 1]
ax.scatter(win, np.full(len(win), 1.06), marker="|", color="#f9a825", s=60, label="full-moon window (± 1 day)")
ax.set_ylim(-0.08, 1.12); ax.set_title("Moon phase in 2024 (calculated)"); ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.45))
ax.xaxis.set_major_locator(mdates.MonthLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.tight_layout()

## 3. Calendar

Holidays come from two sources, cross-referenced:

- **Official list** (`calendar/prazniki_uradni.csv`): all Slovenian holidays, with a work-free flag.
- **Nager.Date** (`date.nager.at/api/v3`, the API behind nagerholidays.com): public holidays for Slovenia, downloaded per year and cached in `data/raw/calendar/nager_SI_<year>.json`.

The cross-reference (`data/processed/holidays_crossref.csv`) lists every date found in either source. Work-free days are the **union** of official work-free days and Nager *Public* holidays. This catches one-off holidays missing from the official file, such as the Day of Solidarity after the August 2023 floods (2023-08-14).

| Column | Meaning |
|---|---|
| `holiday_workfree` | Work-free public holiday (official ∪ Nager) |
| `martinovo` | St Martin's Day, 11 November (not work-free; wine festivities, relevant for alcohol-related injuries) |
| `holiday` | `holiday_workfree` or `martinovo` |
| `day_after_holiday` | Day after a `holiday` |
| `bridge_day` | Working Monday before a Tuesday work-free holiday, or working Friday after a Thursday one |
| `school_holiday` | Primary-school holiday in Maribor, from the OpenHolidays API (cached in `data/raw/calendar/openholidays_school_SI_<year>.json`). Winter holidays alternate between two regions; Maribor belongs to the eastern regions (SI-VR). Entries only for secondary schools are excluded. |

In [ ]:
NAGER_API = "https://date.nager.at/api/v3/PublicHolidays/{year}/SI"
CAL_DIR = RAW / "calendar"


def load_nager(years) -> pd.DataFrame:
    frames = []
    for y in years:
        f = CAL_DIR / f"nager_SI_{y}.json"
        if not f.exists():
            r = requests.get(NAGER_API.format(year=y), timeout=60)
            r.raise_for_status()
            f.write_text(r.text)
        frames.append(pd.DataFrame(json.loads(f.read_text())))
    n = pd.concat(frames, ignore_index=True)
    n["date"] = pd.to_datetime(n["date"])
    n["public"] = n["types"].apply(lambda t: "Public" in t)
    return n


def build_holidays() -> tuple[set, pd.DataFrame]:
    off = pd.read_csv(CAL_DIR / "prazniki_uradni.csv", sep=";")
    off["date"] = pd.to_datetime(off["DATUM"], format="%d.%m.%Y")
    off["workfree"] = off["DELA_PROST_DAN"].str.strip().str.lower() == "da"
    years = range(START.year, END.year + 1)
    off = off[off["date"].dt.year.isin(years)]
    nag = load_nager(years)
    register("Public holidays (official)", [pd.Timestamp(f"{min(years)}-01-01"), pd.Timestamp(f"{max(years)}-12-31")],
             "exposure", "daily", "Official calendar", "calendar/prazniki_uradni.csv")
    register("Public holidays (Nager.Date)", [pd.Timestamp(f"{min(years)}-01-01"), pd.Timestamp(f"{max(years)}-12-31")],
             "exposure", "daily", "Nager.Date", "calendar/nager_SI_*.json")

    a = off.groupby("date").agg(official_name=("IME_PRAZNIKA", " / ".join), official_workfree=("workfree", "any"))
    b = nag.groupby("date").agg(nager_name=("localName", " / ".join), nager_types=("types", lambda t: ", ".join(sorted({x for l in t for x in l}))),
                                nager_public=("public", "any"))
    x = a.join(b, how="outer")
    x["in_official"], x["in_nager"] = x["official_name"].notna(), x["nager_name"].notna()
    x["official_workfree"] = x["official_workfree"].fillna(False).astype(bool)
    x["nager_public"] = x["nager_public"].fillna(False).astype(bool)
    x["workfree"] = x["official_workfree"] | x["nager_public"]
    x["agree_workfree"] = x["official_workfree"] == x["nager_public"]
    x.to_csv(OUTPUTS / "holidays_crossref.csv")
    return set(x.index[x["workfree"]]), x


workfree_days, holidays_crossref = build_holidays()
print(f"Dates in both sources: {int((holidays_crossref['in_official'] & holidays_crossref['in_nager']).sum())}; "
      f"official only: {int((~holidays_crossref['in_nager']).sum())}; Nager only: {int((~holidays_crossref['in_official']).sum())}")
print("Disagreement on work-free days:")
holidays_crossref.loc[~holidays_crossref["agree_workfree"], ["official_name", "official_workfree", "nager_name", "nager_types"]]

In [ ]:
OPENHOLIDAYS_API = "https://openholidaysapi.org/SchoolHolidays"
SCHOOL_REGION = "SI-VR"  # winter holidays: Maribor (SI-VR-PD-MB) belongs to the eastern regions


def load_school_holidays() -> tuple[set, pd.DataFrame]:
    """Primary-school holidays for Maribor from OpenHolidays (one request per year; the API allows <= 1095 days)."""
    rows = []
    for y in range(START.year, END.year + 1):
        f = CAL_DIR / f"openholidays_school_SI_{y}.json"
        if not f.exists():
            r = requests.get(OPENHOLIDAYS_API, params=dict(countryIsoCode="SI", languageIsoCode="SL",
                                                          validFrom=f"{y}-01-01", validTo=f"{y}-12-31"), timeout=60)
            r.raise_for_status()
            f.write_text(r.text)
        for h in json.loads(f.read_text()):
            regions = [s["code"] for s in h.get("subdivisions", [])]
            comment = " ".join(c["text"] for c in (h.get("comment") or []))
            rows.append(dict(start=h["startDate"], end=h["endDate"], name=h["name"][0]["text"], nationwide=h["nationwide"],
                             regions=",".join(regions), comment=comment))
    s = pd.DataFrame(rows).drop_duplicates()
    s[["start", "end"]] = s[["start", "end"]].apply(pd.to_datetime)
    # Maribor primary schools: nationwide or eastern-region entries, excluding entries only for secondary schools
    s["applies"] = (s["nationwide"] | s["regions"].str.contains(SCHOOL_REGION)) & ~s["comment"].str.contains("srednje šole")
    days = {d for _, r in s[s["applies"]].iterrows() for d in pd.date_range(r["start"], r["end"], freq="D")}
    register("School holidays (OpenHolidays)", [pd.Timestamp(f"{START.year}-01-01"), pd.Timestamp(f"{END.year}-12-31")],
             "exposure", "daily", "OpenHolidays API", "calendar/openholidays_school_SI_*.json")
    return days, s


school_days, school_holidays = load_school_holidays()
school_holidays[school_holidays["applies"]].sort_values("start")[["start", "end", "name", "regions", "comment"]]

In [ ]:
# COVID-19 in Slovenia (official dates):
# 2020-03-12 epidemic declared; 2022-02-21 most restrictions and the PCT (recovered/vaccinated/tested) rule lifted,
# except in health and social care; 2023-04-01 COVID-19 treated like other respiratory infections.
COVID_PERIOD = ("2020-03-12", "2023-03-31")        # pandemic period (preregistered adjustment), ends with the special rules
COVID_RESTRICTIONS = ("2020-03-12", "2022-02-20")  # period of general restrictions (sensitivity: two-phase adjustment)


def build_calendar() -> pd.DataFrame:
    cal = pd.DataFrame(index=DAYS)
    idx = cal.index
    cal["dow"] = idx.day_of_week  # 0 = Monday
    cal["month"] = idx.month
    cal["year"] = idx.year
    cal["holiday_workfree"] = idx.isin(workfree_days).astype(int)
    cal["martinovo"] = ((idx.month == 11) & (idx.day == 11)).astype(int)
    cal["holiday"] = (cal["holiday_workfree"] | cal["martinovo"]).astype(int)
    cal["day_after_holiday"] = cal["holiday"].shift(1, fill_value=0)
    off = cal["holiday_workfree"].astype(bool) | (cal["dow"] >= 5)
    nxt = pd.Series(idx + pd.Timedelta(days=1)).isin(workfree_days).values
    prv = pd.Series(idx - pd.Timedelta(days=1)).isin(workfree_days).values
    cal["bridge_day"] = (~off & (((cal["dow"] == 0) & nxt) | ((cal["dow"] == 4) & prv))).astype(int)
    cal["school_holiday"] = idx.isin(school_days).astype(int)
    cal["covid"] = ((idx >= COVID_PERIOD[0]) & (idx <= COVID_PERIOD[1])).astype(int)
    cal["covid_restrictions"] = ((idx >= COVID_RESTRICTIONS[0]) & (idx <= COVID_RESTRICTIONS[1])).astype(int)
    cal["time"] = np.arange(len(cal))
    return cal


calendar = build_calendar()
calendar[["holiday_workfree", "martinovo", "holiday", "bridge_day", "school_holiday"]].groupby(calendar["year"]).sum()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.8))
rows = [("holiday_workfree", "work-free holiday", "C3"), ("martinovo", "Martinovo (11 Nov)", "#8e24aa"),
        ("school_holiday", "school holiday", "C2"),
        ("bridge_day", "bridge day", "C1"), ("covid", "COVID period", "C7")]
for y, (col, label, c) in enumerate(rows[::-1]):
    d = calendar.index[calendar[col] == 1]
    ax.scatter(d, np.full(len(d), y), marker="|", s=120, color=c)
ax.set_yticks(range(len(rows)), [r[1] for r in rows[::-1]])
ax.set_ylim(-0.6, len(rows) - 0.4); ax.set_title("Calendar indicators")
study_span(ax); fig.tight_layout()

#### COVID-19 restrictions (Oxford stringency index)

The 0/1 `covid` indicator (pandemic period, 2020-03-12 to 2023-03-31: from the declaration of the epidemic until COVID-19 was treated like other respiratory infections) is 1 for the first 27 months of the panel and partly collinear with the time trend. `covid_restrictions` (to 2022-02-20, when most restrictions and the PCT rule were lifted outside health and social care) marks the first phase, for a two-phase sensitivity adjustment. As an alternative adjustment, the **Oxford COVID-19 Government Response Tracker** (OxCGRT, Blavatnik School of Government, CC BY 4.0) gives a daily **stringency index** for Slovenia (0–100; school and workplace closures, event cancellations, gathering limits, public transport, stay-at-home orders, travel limits and public information campaigns). The national file is downloaded once; only the Slovenia rows are cached in `data/raw/oxcgrt/`.

- `covid_stringency`: as published; missing after the dataset ends (2022-12-31).
- `covid_stringency_filled`: the last published value carried forward after the end [assumption: no further change]. The last value (11.1) corresponds to one of nine indicators at its maximum, most likely the public information campaign, not a restriction.

In [ ]:
OXCGRT_URL = "https://raw.githubusercontent.com/OxCGRT/covid-policy-dataset/main/data/OxCGRT_compact_national_v1.csv"
OXCGRT_DIR = RAW / "oxcgrt"
OXCGRT_DIR.mkdir(parents=True, exist_ok=True)


def load_oxcgrt() -> pd.DataFrame:
    f = OXCGRT_DIR / "oxcgrt_slovenia.csv"
    if not f.exists():
        r = requests.get(OXCGRT_URL, timeout=600)
        r.raise_for_status()
        full = pd.read_csv(io.BytesIO(r.content), low_memory=False)
        si = full[(full["CountryCode"] == "SVN") & (full["Jurisdiction"] == "NAT_TOTAL")]
        assert len(si) > 0, "no Slovenia rows in OxCGRT"
        si.to_csv(f, index=False)
        print(f"downloaded OxCGRT: {len(si)} Slovenia rows")
    si = pd.read_csv(f, low_memory=False)
    si["date"] = pd.to_datetime(si["Date"].astype(str), format="%Y%m%d")
    s = si.set_index("date")["StringencyIndex_Average"].sort_index()
    assert s.index.is_unique
    register("COVID-19 restrictions (OxCGRT)", s.index, "exposure", "daily", "Oxford COVID-19 Government Response Tracker", "oxcgrt/oxcgrt_slovenia.csv")
    out = to_daily(s.rename("covid_stringency").to_frame())
    last = s.dropna().index.max()
    out["covid_stringency_filled"] = out["covid_stringency"].where(out.index <= last, s.loc[last])
    print(f"Stringency index until {last:%Y-%m-%d}; last value {s.loc[last]:.1f} carried forward in covid_stringency_filled")
    return out


oxcgrt = load_oxcgrt()
oxcgrt.resample("QS").mean().round(1).T

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(oxcgrt.index, oxcgrt["covid_stringency_filled"], color="0.6", lw=1, ls="--", label="carried forward (assumption)")
ax.plot(oxcgrt.index, oxcgrt["covid_stringency"], color="C3", lw=1.2, label="stringency index (OxCGRT)")
ax.fill_between(calendar.index, 0, 100, where=calendar["covid"] == 1, color="0.5", alpha=0.1, step="mid", label="covid = 1")
ax.set_ylim(0, 100); ax.set_title("COVID-19 restrictions in Slovenia: stringency index vs the 0/1 covid indicator"); ax.legend(loc="upper right")
study_span(ax); fig.tight_layout()

## 4. Data overview before joining

**Time frames:** bars show the full raw coverage of each dataset. The grey band is the study period. Datasets that start before the left edge have their first date written next to the bar.

**Completeness:** the share of study-period days on which each dataset has a value, per month and overall. Each dataset is represented by its key column (`KEY_COLUMNS`). Weekly ERVISS and monthly Google Trends are counted after they are spread to days.

### Look-ahead check

For each low-frequency source, values that become available after a cut-off date are replaced with nonsense, and the real-time (`_rt`) columns are recomputed. If any `_rt` value on or before the cut-off changes, that column uses information that was not yet published, and the check fails.

In [ ]:
def availability(spec) -> pd.DatetimeIndex:
    if spec["kind"] == "period":
        return pd.DatetimeIndex(spec["ref_end"]) + pd.Timedelta(days=spec["lag_days"])
    return spec["values"].index + pd.Timedelta(days=spec["lag_days"])


def rt_columns(spec, values) -> pd.DataFrame:
    if spec["kind"] == "period":
        out = period_versions(values, spec["ref_start"], spec["ref_end"], spec["anchor"], spec["lag_days"], spec["max_carry_days"])
    else:
        out = sample_versions(values, spec["lag_days"], spec["max_carry_days"])
    return out.filter(regex="_rt$")


results = []
for name, spec in LOWFREQ_INPUTS.items():
    avail = availability(spec)
    base = rt_columns(spec, spec["values"])
    in_study = pd.Series(avail[(avail >= START) & (avail <= END)])
    for cutoff in in_study.quantile([0.25, 0.5, 0.75]).dt.normalize():
        perturbed = spec["values"].copy()
        later = np.asarray(avail > cutoff)
        perturbed.loc[later, :] = perturbed.loc[later, :].to_numpy(dtype=float) * 10 + 999
        test = rt_columns(spec, perturbed)
        before = base.loc[:cutoff].dropna(how="all")
        ok = before.equals(test.loc[before.index])
        results.append((name, cutoff.date(), int(later.sum()), len(before), ok))
check = pd.DataFrame(results, columns=["source", "cutoff", "values_perturbed", "rt_days_compared", "no_look_ahead"])
assert (check["values_perturbed"] > 0).all() and (check["rt_days_compared"] > 0).all(), "check is empty"
assert check["no_look_ahead"].all(), check[~check["no_look_ahead"]]

# Control: the interpolated columns DO look ahead, so the same test must fail for them
spec = LOWFREQ_INPUTS["erviss"]
avail = availability(spec)
cutoff = pd.Series(avail[(avail >= START) & (avail <= END)]).quantile(0.5).normalize()
perturbed = spec["values"].copy()
later = np.asarray(avail > cutoff)
perturbed.loc[later, :] = perturbed.loc[later, :].to_numpy(dtype=float) * 10 + 999
args = (spec["ref_start"], spec["ref_end"], spec["anchor"], spec["lag_days"], spec["max_carry_days"])
a_ = period_versions(spec["values"], *args).filter(regex="_interp$").loc[:cutoff]
b_ = period_versions(perturbed, *args).filter(regex="_interp$").loc[:cutoff]
assert not a_.equals(b_), "control failed: the test cannot detect look-ahead"
print("Control: interpolated ERVISS columns change before the cut-off, as expected (they look ahead).")
check

In [ ]:
KEY_COLUMNS = {
    "HIS": ("outcomes", "arr_all_mts_all"),
    "eTriaža": ("outcomes", "arr_ambulance"),
    "Weather (ARSO)": ("arso_weather", "temp_mean"),
    "Reanalysis weather (ERA5-Land / ERA5)": ("era5", "era5_rh_mean"),
    "Weather warnings (ARSO text)": ("arso_warnings", "warn_any"),
    "Weather warnings (MeteoAlarm)": ("meteoalarm", "ma_warn_level"),
    "Air SI0059A PM10": ("arso_air", "pm10"),
    "Air SI0059A PM25": ("arso_air", "pm25"),
    "Air SI0059A NO2": ("arso_air", "no2"),
    "Air SI0059A O3": ("arso_air", "o3"),
    "Air E407 MB Titova PM10": ("arso_air", "pm10_titova"),
    "Air E407 MB Titova PM25": ("arso_air", "pm25_titova"),
    "Air E408 MB Vrbanski PM10": ("arso_air", "pm10_vrbanski"),
    "Air E408 MB Vrbanski PM25": ("arso_air", "pm25_vrbanski"),
    "ERVISS (Slovenia)": ("erviss", "ili_rate"),
    "Influenza index (PCR + ERVISS)": ("flu_index", "flu_index"),
    "PCR testing (Maribor)": ("pcr", "pcr_sars"),
    "COVID-19 cases Štajerska (Sledilnik)": ("covid", "covid_cases_stajerska"),
    "COVID-19 patients UKC Maribor (Sledilnik)": ("covid", "covid_ukcmb_inpatients"),
    "COVID-19 wastewater Maribor (Sledilnik)": ("covid", "covid_sewage_mb"),
    "Pollen (Maribor)": ("pollen", "pollen_total"),
    "Google Trends": ("google_trends", "gt_urgenca_maribor"),
    "SBI TOP": ("sbi_top", "sbi_close"),
    "NDVI maribor": ("ndvi", "ndvi_maribor"),
    "NDVI stajerska": ("ndvi", "ndvi_stajerska"),
    "EMS dispatch (Maribor)": ("ems", "ems_calls"),
    "Football fixtures (MNZ Maribor)": ("football", "football_matches"),
    "Hospital occupancy": ("hospital_capacity", "hosp_occupancy"),
    "Moon phase (calculated + USNO)": ("moon", "moon_illumination"),
    "Public holidays (official)": ("calendar", "holiday_workfree"),
    "Public holidays (Nager.Date)": ("calendar", "holiday_workfree"),
    "School holidays (OpenHolidays)": ("calendar", "school_holiday"),
    "COVID-19 restrictions (OxCGRT)": ("oxcgrt", "covid_stringency"),
}
frames = {k: globals()[k] for k in {f for f, _ in KEY_COLUMNS.values()} if globals().get(k) is not None}
avail = pd.DataFrame({name: frames[f][c] for name, (f, c) in KEY_COLUMNS.items() if f in frames and c in frames[f]}).notna()
# SBI only trades on working days: count a day as covered if the market was open or it was a non-trading day inside the series
if "SBI TOP" in avail:
    avail["SBI TOP"] = sbi_top["sbi_close"].ffill().notna()

cov = pd.DataFrame(COVERAGE).T
cov.index.name = "dataset"
for c in ["raw_start", "raw_end"]:
    cov[c] = pd.to_datetime(cov[c])
cov["study_start"] = cov["raw_start"].clip(lower=START)
cov["study_end"] = cov["raw_end"].clip(upper=END)
cov.loc[cov["study_start"] > cov["study_end"], ["study_start", "study_end"]] = pd.NaT
cov["days_in_study"] = ((cov["study_end"] - cov["study_start"]).dt.days + 1).fillna(0).astype(int)
cov["completeness_pct"] = (avail.mean() * 100).round(1).reindex(cov.index)
cov["key_column"] = [KEY_COLUMNS.get(i, ("", ""))[1] for i in cov.index]
order = {"outcome": 0, "exposure": 1, "exploratory": 2}
cov = cov.sort_values(["group", "raw_start"], key=lambda s: s.map(order) if s.name == "group" else s)

cov_out = cov.copy()
for c in ["raw_start", "raw_end", "study_start", "study_end"]:
    cov_out[c] = pd.to_datetime(cov_out[c]).dt.date
cov_out[["group", "provider", "resolution", "files", "key_column", "raw_start", "raw_end",
         "study_start", "study_end", "days_in_study", "completeness_pct"]].to_csv(OUTPUTS / "dataset_coverage.csv")
print(f"Saved {(OUTPUTS / 'dataset_coverage.csv').relative_to(ROOT)}")
cov_out[["group", "resolution", "raw_start", "raw_end", "days_in_study", "completeness_pct"]]

In [ ]:
group_color = {"outcome": "#c62828", "exposure": "#1565c0", "exploratory": "#2e7d32"}
x0, x1 = pd.Timestamp("2018-01-01"), pd.Timestamp("2027-01-01")
names = list(cov.index)
fig, ax = plt.subplots(figsize=(12, 0.32 * len(names) + 1.2))
ax.axvspan(START, END, color="0.5", alpha=0.12, lw=0)
for i, (name, r) in enumerate(cov.iterrows()):
    s, e = max(r["raw_start"], x0), min(r["raw_end"], x1)
    ax.barh(i, (e - s).days, left=s, height=0.6, color=group_color[r["group"]], alpha=0.85)
    if r["raw_start"] < x0:
        ax.text(x0, i, f"◀ from {r['raw_start']:%Y-%m}", va="center", ha="left", fontsize=7, color="white")
    if r["raw_end"] > x1:
        ax.text(x1, i, f"to {r['raw_end']:%Y-%m} ▶", va="center", ha="right", fontsize=7, color="white")
    pct = r["completeness_pct"]
    ax.text(x1 + pd.Timedelta(days=20), i, "" if pd.isna(pct) else f"{pct:.0f}%", va="center", fontsize=7.5)
ax.set_yticks(range(len(names)), names); ax.invert_yaxis()
ax.set_xlim(x0, x1); ax.grid(axis="y", visible=False)
ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in group_color.values()]
ax.legend(handles, group_color.keys(), ncol=4, loc="lower left", bbox_to_anchor=(0, 1.0))
ax.set_title(f"Dataset time frames (grey = study period {START:%Y-%m-%d} – {END:%Y-%m-%d}); right: completeness in study period",
             loc="left", pad=22)
fig.tight_layout()

In [ ]:
monthly_avail = avail.resample("MS").mean().T.reindex([n for n in cov.index if n in avail])
fig, ax = plt.subplots(figsize=(12, 0.32 * len(monthly_avail) + 1.4))
im = ax.imshow(monthly_avail.values * 100, aspect="auto", cmap="RdYlGn", vmin=0, vmax=100, interpolation="nearest",
               extent=[mdates.date2num(monthly_avail.columns[0]), mdates.date2num(monthly_avail.columns[-1] + pd.offsets.MonthEnd(0)),
                       len(monthly_avail) - 0.5, -0.5])
ax.set_yticks(range(len(monthly_avail)), [f"{n}  ({avail[n].mean():.0%})" for n in monthly_avail.index])
ax.xaxis_date(); ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(False)
fig.colorbar(im, ax=ax, label="% of days with data", pad=0.01)
ax.set_title("Data completeness per month in the study period (key column per dataset)")
fig.tight_layout()

## 5. Merge

In [ ]:
parts = {
    "outcomes": outcomes, "calendar": calendar, "oxcgrt": oxcgrt,
    "arso_weather": arso_weather, "era5": era5, "arso_warnings": arso_warnings, "meteoalarm": meteoalarm, "arso_air": arso_air,
    "erviss": erviss, "pcr": pcr, "flu_index": flu_index, "covid": covid, "pollen": pollen, "google_trends": google_trends,
    "sbi_top": sbi_top, "ndvi": ndvi,
    "ems": ems, "football": football, "hospital_capacity": hospital_capacity, "moon": moon,
}
included = {k: v for k, v in parts.items() if v is not None}
print("Skipped:", ", ".join(k for k, v in parts.items() if v is None) or "-")

panel = pd.concat(included.values(), axis=1)
assert panel.index.equals(DAYS)
assert not panel.columns.duplicated().any(), panel.columns[panel.columns.duplicated()].tolist()

# Columns without information over the study period. Dropped explicitly (not by a silent rule): if the data change
# and another column becomes constant, or one of these varies, the run stops so the list can be reviewed.
CONSTANT_DROP = {
    "att_unlinked_etriaza",      # 0 wherever eTriaža exists: linkage is complete; after 2026-04-22 eTriaža is missing
    "adm_fasttrack", "adm_fasttrack_pct",  # Hitri pregledi never admits; patients are referred to KPP/INP first
    "warn_strict_fire_danger",   # no strict fire-danger warning in the study period
}
CONSTANT_KEEP = {"arr_inp_mts1"}  # INP never sees red (MTS 1); kept so every stream has the same set of triage series
constant = {c for c in panel.columns if panel[c].nunique(dropna=True) <= 1}
assert constant == CONSTANT_DROP | CONSTANT_KEEP, f"constant columns changed: {sorted(constant ^ (CONSTANT_DROP | CONSTANT_KEEP))}"
panel = panel.drop(columns=sorted(CONSTANT_DROP))
included = {k: v.drop(columns=[c for c in v.columns if c in CONSTANT_DROP]) for k, v in included.items()}
print(f"Dropped {len(CONSTANT_DROP)} constant columns: {sorted(CONSTANT_DROP)}")
print(f"{panel.shape[0]} days x {panel.shape[1]} columns")

### Data flow (Sankey)

How the sources feed the daily panel: **provider → dataset → role in the analysis → daily panel**. Link width is the number of variables (columns) a dataset contributes. Hover over a link to see its columns.

In [ ]:
DATASET_INFO = {  # panel part -> (label, provider, role)
    "outcomes": ("HIS + eTriaža", "UKC Maribor", "Outcome"),
    "arso_weather": ("Weather", "ARSO", "Exposure"),
    "era5": ("Reanalysis weather (ERA5)", "Copernicus via Open-Meteo", "Exposure"),
    "arso_warnings": ("Weather warnings (text)", "ARSO", "Exploratory"),
    "meteoalarm": ("Official warnings (2025+)", "MeteoAlarm", "Exposure"),
    "arso_air": ("Air quality (Maribor)", "ARSO / EEA", "Exposure"),
    "erviss": ("Respiratory illness (ERVISS)", "ECDC", "Exposure"),
    "pcr": ("PCR testing (Maribor)", "NLZOH, Maribor", "Exposure"),
    "flu_index": ("Influenza index (PCR + ERVISS)", "NLZOH + ECDC (derived)", "Exposure"),
    "covid": ("COVID-19 Štajerska", "Sledilnik", "Exposure"),
    "google_trends": ("Google Trends", "Google", "Exposure"),
    "sbi_top": ("SBI TOP", "Ljubljana Stock Exchange", "Exposure"),
    "calendar": ("Calendar", "Official calendar + Nager.Date + OpenHolidays", "Exposure"),
    "oxcgrt": ("COVID-19 restrictions (OxCGRT)", "Oxford Government Response Tracker", "Exposure"),
    "pollen": ("Pollen", "Pollen station Maribor", "Exploratory"),
    "ndvi": ("NDVI", "Copernicus Sentinel-2", "Exploratory"),
    "ems": ("EMS dispatch", "EMS Maribor", "Exploratory"),
    "football": ("Football fixtures", "NZS", "Exploratory"),
    "hospital_capacity": ("Hospital occupancy", "UKC Maribor", "Exploratory"),
    "moon": ("Moon phase", "Calculated + USNO", "Exploratory"),
}
ROLE_COLOR = {"Outcome": "#c62828", "Exposure": "#1565c0", "Exploratory": "#2e7d32"}
assert set(included) <= set(DATASET_INFO), set(included) - set(DATASET_INFO)


def rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    return f"rgba({int(h[0:2], 16)}, {int(h[2:4], 16)}, {int(h[4:6], 16)}, {alpha})"


def column_list(cols, n=12):
    cols = list(cols)
    return "<br>".join(cols[:n]) + (f"<br>… and {len(cols) - n} more" if len(cols) > n else "")


rows = pd.DataFrame(
    [(k, *DATASET_INFO[k], list(v.columns)) for k, v in included.items()],
    columns=["part", "dataset", "provider", "role", "columns"],
)
rows["n"] = rows["columns"].str.len()
# Group datasets by role (outcome, exposure, exploratory) so flows do not cross
rows = rows.sort_values("role", key=lambda r: r.map(list(ROLE_COLOR).index), kind="stable").reset_index(drop=True)
panel_label = f"Daily panel<br>{panel.shape[0]:,} days × {panel.shape[1]} columns"

nodes = (
    [f"{p}" for p in rows["provider"].unique()]
    + [f"{d} ({n})" for d, n in zip(rows["dataset"], rows["n"])]
    + [f"{r} ({rows.loc[rows['role'] == r, 'n'].sum()})" for r in ROLE_COLOR if r in set(rows["role"])]
    + [panel_label]
)
idx = {name: i for i, name in enumerate(nodes)}
role_node = {r: f"{r} ({rows.loc[rows['role'] == r, 'n'].sum()})" for r in ROLE_COLOR if r in set(rows["role"])}

node_color = []
for name in nodes:
    if name == panel_label:
        node_color.append("#424242")
    elif name in role_node.values():
        node_color.append(ROLE_COLOR[name.split(" (")[0]])
    else:
        node_color.append("#9e9e9e")

src, tgt, val, col, hover = [], [], [], [], []
for _, r in rows.iterrows():
    d = f"{r['dataset']} ({r['n']})"
    c = ROLE_COLOR[r["role"]]
    for a, b in [(r["provider"], d), (d, role_node[r["role"]])]:
        src.append(idx[a]); tgt.append(idx[b]); val.append(r["n"]); col.append(rgba(c, 0.35))
        hover.append(column_list(r["columns"]))
for role, name in role_node.items():
    cols = [c for cs in rows.loc[rows["role"] == role, "columns"] for c in cs]
    src.append(idx[name]); tgt.append(idx[panel_label]); val.append(len(cols)); col.append(rgba(ROLE_COLOR[role], 0.35))
    hover.append(column_list(cols))
assert sum(v for s, v in zip(src, val) if nodes[s] in role_node.values()) == panel.shape[1]

# Fixed node positions: each column is stacked top to bottom in the order above
def stack(names, sizes, x, gap=0.025):
    total = sum(sizes)
    scale = (1 - gap * (len(names) - 1)) / total
    y, pos = 0.0, {}
    for n, v in zip(names, sizes):
        pos[n] = (x, max(0.001, min(0.999, y + v * scale / 2)))
        y += v * scale + gap
    return pos


providers = list(dict.fromkeys(rows["provider"]))
pos = {}
pos |= stack(providers, [rows.loc[rows["provider"] == p, "n"].sum() for p in providers], 0.001)
pos |= stack([f"{d} ({n})" for d, n in zip(rows["dataset"], rows["n"])], list(rows["n"]), 0.33)
pos |= stack(list(role_node.values()), [rows.loc[rows["role"] == r, "n"].sum() for r in role_node], 0.67)
pos[panel_label] = (0.999, 0.5)

fig = go.Figure(go.Sankey(
    arrangement="fixed",
    node=dict(label=nodes, color=node_color, pad=14, thickness=16, line=dict(width=0),
              x=[pos[n][0] for n in nodes], y=[pos[n][1] for n in nodes]),
    link=dict(source=src, target=tgt, value=val, color=col, customdata=hover,
              hovertemplate="%{source.label} → %{target.label}<br><b>%{value} columns</b><br>%{customdata}<extra></extra>"),
))
fig.update_layout(
    title=dict(text="Data flow: provider → dataset → role → daily panel (width = number of columns)", x=0.01),
    font=dict(size=11), height=720, margin=dict(l=10, r=10, t=50, b=10),
)
fig.show()

## 6. Column coverage and save

In [ ]:
source_of = {c: k for k, v in included.items() for c in v.columns}
qc = pd.DataFrame({
    "source": pd.Series(source_of),
    "first": panel.apply(lambda s: s.first_valid_index()).dt.date,
    "last": panel.apply(lambda s: s.last_valid_index()).dt.date,
    "pct_missing": (panel.isna().mean() * 100).round(1),
}).loc[panel.columns]

out = PROCESSED / "daily_panel.parquet"
panel.to_parquet(out)
qc.to_csv(OUTPUTS / "daily_panel_columns.csv")
print(f"Saved {out.relative_to(ROOT)} and {(OUTPUTS / 'daily_panel_columns.csv').relative_to(ROOT)}")
with pd.option_context("display.max_rows", 200):
    display(qc)

## 7. Codebook

Every column of the panel is defined here and written to `data/processed/daily_panel_codebook.csv`, next to the panel. The run stops if a column has no definition.

**`use`** says how a column is meant to be used, so that the main analysis is fixed before any results are seen:

| use | Meaning |
|---|---|
| `primary` | Outcome, exposure or adjustment variable of the **primary** models for H1–H3 |
| `secondary` | Preregistered secondary analyses (other streams, triage levels, other preregistered exposures) |
| `sensitivity` | Alternative definition of a primary or secondary variable, for sensitivity analyses |
| `forecasting` | Real-time (`_rt`) versions for the forecasting module; not for explanatory models |
| `exploratory` | Not preregistered |
| `descriptive` | Too rare or not suitable for modelling (e.g. daily percentages); describe only |
| `technical` | Flags, indicators of structural breaks and helper columns |

**Availability for forecasting.** `forecast_lag_days` is the number of days after the date until a value is known, with the forecast made at the end of day *t*: a forecast from day *t* may use a column's value for day *d* only if *d* + `forecast_lag_days` ≤ *t*. Negative = known in advance (calendar, moon, fixtures); empty = not a real-time series (use its `_rt` version, or do not use it for forecasting). `available` explains each value; lags marked [assumption] should be checked against the providers.

**Primary models (fixed 2026-11-09; see `docs/deviations_log.md`):** H1 = `arr_all_mts_all` ~ `temp_mean`, also adjusted for influenza (`flu_index`); H2 = `arr_medical_mts_all` ~ `ili_rate`, adjusted for `hp_referral_era`; H3 = `arr_medical_mts_all` ~ `pm10`, adjusted for `temp_mean` and `hp_referral_era`; all adjusted for `time` (trend and seasonality), `dow`, `holiday` and `covid`. Other stream-level models add `hp_referral_era`; triage-specific models add `triage_break_2024_06`. **Negative-control outcomes** for H2 and H3: injuries by unit (`arr_injury_mts_all`, KPP) and by diagnosis (`arr_dx_injury_icd`, ICD-10 S/T); the H2/H3 models are refitted with each.

In [ ]:
NAME = {  # readable names for regex groups
    "flu": "influenza", "rsv": "RSV", "sars": "SARS-CoV-2", "flu_a": "influenza A", "flu_b": "influenza B",
    "titova": "Titova (traffic)", "vrbanski": "Vrbanski plato (urban background)", "max": "maximum", "min": "minimum", "mean": "mean",
    "maribor": "the Maribor box (15.55-15.75 E, 46.50-46.62 N)", "stajerska": "Štajerska (Podravska + Savinjska)",
    "high_temp": "high-temperature", "low_temp": "low-temperature", "snow_ice": "snow/ice", "forest_fire": "forest-fire",
}
STREAM_NAME = {"all": "all units", "injury": "KPP (Enota za poškodbe)", "disease": "Enota za bolezni", "inp": "INP (Internistična nujna pomoč)",
               "fasttrack": "Hitri pregledi only", "medical": "medical stream (Enota za bolezni + INP)"}
DX_NOTE = "Hitri pregledi records no diagnoses: covers only episodes seen in KPP, INP or Enota za bolezni."
EP = (f"Episodes (patients): records of the same patient within {EPISODE_WINDOW_H} h of the previous discharge count once, "
      "dated by the first arrival.")

# (regex on the base column name, unit, definition (may use {0}, {1} for regex groups), use, hypothesis, note). First match wins.
CODEBOOK_RULES = [
    # outcomes
    (r"arr_all_mts_all", "episodes/day", "All ED episodes, all triage categories. " + EP, "primary", "H1", ""),
    (r"arr_medical_mts_all", "episodes/day", "Episodes ending in Enota za bolezni or INP (medical stream). " + EP, "primary", "H2, H3",
     "Replaces the preregistered 'disease arrivals': routing between the two units changed in 2021-23. Include hp_referral_era"),
    (r"arr_all_mts_all_w(\d+)h", "episodes/day", "As arr_all_mts_all, with a {0}-hour linking window", "sensitivity", "H1", ""),
    (r"arr_(disease|inp)_mts_all", "episodes/day", "Episodes ending in {0} (single unit)", "sensitivity", "H2, H3", "Include hp_referral_era"),
    (r"arr_injury_mts_all", "episodes/day", "Episodes ending in KPP (Enota za poškodbe)", "secondary", "H2, H3 (negative-control outcome)",
     "Negative-control outcome for H2 and H3 (not for weather). Include hp_referral_era"),
    (r"arr_fasttrack_mts_all", "episodes/day", "Episodes ending in Hitri pregledi only", "secondary", "", "Include hp_referral_era"),
    (r"arr_(all|injury|disease|inp|fasttrack|medical)_mts([1-5])", "episodes/day",
     "Episodes of {0} whose most urgent Manchester category is {1} (1 red, 2 orange, 3 yellow, 4 green, 5 blue)", "secondary", "",
     "Include triage_break_2024_06"),
    (r"ref_hp_(\w+)", "referrals/day", "Referrals from Hitri pregledi to {0}, dated by the arrival at Hitri pregledi", "secondary", "",
     f"Collapse on {HP_REFERRAL_BREAK:%Y-%m-%d} (hp_referral_era)"),
    (r"direct_(\w+)", "episodes/day", "Episodes whose first contact was {0} (direct arrival)", "secondary", "", ""),
    (r"att_(\w+)", "records/day", "Unit attendances of {0}: HIS records, not linked into episodes (workload)", "descriptive", "", ""),
    (r"adm_(\w+)_pct", "%", "Share of episodes in {0} that were admitted to hospital", "descriptive", "",
     "Model admission counts with the number of episodes as offset instead"),
    (r"adm_(\w+)", "admissions/day", "Hospital admissions from {0} (unit of the admitting record)", "secondary", "", ""),
    (r"arr_dx_R07", "episodes/day", "Episodes with final diagnosis R07 (chest pain)", "secondary", "", "Outcome for the SBI TOP analysis. " + DX_NOTE),
    (r"arr_dx_F41", "episodes/day", "Episodes with final diagnosis F41 (anxiety, panic)", "descriptive", "", "Rare (see n_nonzero_days). " + DX_NOTE),
    (r"arr_dx_resp", "episodes/day", "Episodes with a respiratory final diagnosis (ICD-10 J)", "secondary", "", DX_NOTE),
    (r"arr_dx_circ", "episodes/day", "Episodes with a circulatory final diagnosis (ICD-10 I)", "secondary", "", DX_NOTE),
    (r"arr_dx_injury_icd", "episodes/day", "Episodes with an injury or poisoning final diagnosis (ICD-10 S, T)", "secondary", "H2, H3 (negative-control outcome)",
     "Negative-control outcome for air pollution and influenza only, not for weather. " + DX_NOTE),
    (r"arr_dx_missing", "episodes/day", "Episodes without a final diagnosis (mostly Hitri pregledi)", "technical", "", ""),
    (r"arr_died_in_ed", "episodes/day", "Episodes with death in the ED", "descriptive", "", "Small numbers"),
    (r"arr_age65plus", "episodes/day", "Episodes of patients aged 65 or more", "secondary", "", ""),
    (r"arr_age_under18", "episodes/day", "Episodes of patients under 18", "secondary", "", ""),
    (r"arr_ambulance", "episodes/day", "Episodes arriving by emergency ambulance or reanimobil (eTriaža)", "secondary", "", "Missing where eTriaža is missing (see valid_to)"),
    (r"etr_count", "records/day", "eTriaža records per day", "technical", "", "Used for the HIS gap check"),
    (r"his_gap", "0/1", "1 = HIS extraction gap (< 85% of eTriaža records); all outcomes missing", "technical", "", ""),
    (r"hp_referral_era", "0/1", f"1 before {HP_REFERRAL_BREAK:%Y-%m-%d}, when Hitri pregledi still referred many patients (~20/day)", "primary", "H2, H3 (adjustment)", "Structural-break indicator: adjustment in the H2/H3 primary models (medical stream) and in all stream-level models"),
    (r"triage_break_2024_06", "0/1", f"1 from {TRIAGE_BREAK:%Y-%m-%d} (shift from yellow to green triage in INP and Hitri pregledi)", "technical", "",
     "Structural-break indicator for triage-specific models"),
    # calendar
    (r"dow", "0-6", "Day of week (0 = Monday)", "primary", "H1, H2, H3", "Adjustment (preregistered)"),
    (r"holiday", "0/1", "Work-free public holiday (official list and Nager.Date) or Martinovo (11 Nov)", "primary", "H1, H2, H3",
     "Adjustment (preregistered as public holidays; Martinovo added)"),
    (r"holiday_workfree", "0/1", "Work-free public holiday only (official list and Nager.Date)", "sensitivity", "H1, H2, H3", ""),
    (r"martinovo", "0/1", "St Martin's Day, 11 November", "secondary", "", ""),
    (r"day_after_holiday", "0/1", "Day after a holiday", "secondary", "", ""),
    (r"bridge_day", "0/1", "Working Monday before a Tuesday holiday, or working Friday after a Thursday holiday", "secondary", "", ""),
    (r"school_holiday", "0/1", "Primary-school holiday in Maribor (OpenHolidays; eastern region for winter holidays)", "secondary", "", ""),
    (r"covid_stringency", "0-100", "Oxford COVID-19 Government Response Tracker stringency index, Slovenia (as published)", "sensitivity", "H1, H2, H3",
     "Alternative to the covid indicator; missing after the dataset ends"),
    (r"covid_stringency_filled", "0-100", "Stringency index with the last published value carried forward after the dataset ends", "sensitivity", "H1, H2, H3",
     "Carried-forward part is an assumption (no further change)"),
    (r"covid", "0/1", f"COVID-19 pandemic period in Slovenia ({COVID_PERIOD[0]} to {COVID_PERIOD[1]}): from the declaration of the epidemic until COVID-19 was treated like other respiratory infections (1 April 2023)", "primary", "H1, H2, H3", "Adjustment only (preregistered); official dates"),
    (r"covid_restrictions", "0/1", f"Period of general COVID-19 restrictions ({COVID_RESTRICTIONS[0]} to {COVID_RESTRICTIONS[1]}); on 21 February 2022 most restrictions and the PCT rule were lifted except in health and social care", "sensitivity", "H1, H2, H3", "Two-phase adjustment together with covid"),
    (r"time", "days", f"Days since {START:%Y-%m-%d}, for trend and seasonality splines", "primary", "H1, H2, H3", "Adjustment (preregistered)"),
    (r"month", "1-12", "Calendar month", "technical", "", ""),
    (r"year", "year", "Calendar year", "technical", "", ""),
    # weather (ARSO station)
    (r"temp_mean", "°C", "Daily mean air temperature at 2 m, ARSO Maribor - letališče: (T7 + T14 + 2·T21) / 4 from the 07, 14 and 21 h CET readings (Kämtz)", "primary", "H1; H3 (adjustment)", "Not a 24-hour mean (ARSO climatological definition)"),
    (r"temp_(max|min)", "°C", "Daily {0} air temperature at 2 m, ARSO Maribor - letališče", "sensitivity", "H1", ""),
    (r"temp_range", "°C", "Daily temperature range (max - min)", "secondary", "", ""),
    (r"precip", "mm", "Precipitation from 07:00 CET on the day to 07:00 CET the next day (ARSO raw value of the next day, shifted back)", "secondary", "", "ARSO dates the 24-hour total by the 07:00 CET reading day; shifted so the day holds mostly its own rain. Checked against ERA5 (scripts/check_arso_precip_dating.py)"),
    (r"snow_depth", "cm", "Total snow depth measured at 07:00 CET", "secondary", "", "Morning state, not shifted"),
    (r"snow_new", "cm", "New snow from 07:00 CET on the day to 07:00 CET the next day (ARSO raw value of the next day, shifted back)", "secondary", "", "Same reading-day convention as precipitation (ARSO definition)"),
    (r"sunshine", "h", "Sunshine duration of the day", "secondary", "", ""),
    (r"cloud", "%", "Mean cloud cover of the 07, 14 and 21 h CET observations", "secondary", "", "Partly missing (see pct_missing)"),
    (r"wx_storm", "0/1", "Thunderstorm (nevihta) observed: lightning seen and thunder heard", "secondary", "", ""),
    (r"wx_thunder", "0/1", "Thunder (grmenje) observed", "secondary", "", ""),
    (r"wx_lightning", "0/1", "Lightning (bliskanje) observed", "secondary", "", ""),
    (r"wx_hail", "0/1", "Hail (toča, ice 5-50 mm) observed", "secondary", "", ""),
    (r"wx_gale", "0/1", "Gale (viharni veter): wind >= 8 Beaufort (>= 17.2 m/s) observed", "secondary", "", "Observers also use automatic-station data at night (since 2000)"),
    (r"hot_day", "0/1", "Hot day (ARSO: vroč dan): Tmax >= 30 °C", "secondary", "", ""),
    (r"heatwave", "0/1", "Day within a run of >= 3 consecutive hot days (Tmax >= 30 °C)", "secondary", "", ""),
    (r"frost_day", "0/1", "Frost day (ARSO: hladni dan): Tmin < 0 °C", "secondary", "", ""),
    (r"very_hot_day", "0/1", "Very hot day: Tmax >= 35 °C", "descriptive", "", "Too rare to model"),
    (r"tropical_night", "0/1", "Tropical night (ARSO: topla noč): Tmin >= 20 °C", "descriptive", "", "Too rare to model"),
    (r"ice_day", "0/1", "Ice day (ARSO: ledeni dan): Tmax < 0 °C", "descriptive", "", "Too rare to model"),
    (r"cold_day", "0/1", "Cold day (ARSO: mrzli dan): Tmin <= -10 °C", "descriptive", "", "Too rare to model"),
    # reanalysis
    (r"era5_temp_(mean|max|min)", "°C", "Daily {0} air temperature, ERA5-Land grid cell over Maribor", "sensitivity", "H1", ""),
    (r"era5_apparent_temp_(mean|max)", "°C", "Daily {0} apparent temperature (Steadman, ERA5-Land temperature and humidity, ERA5 wind)", "sensitivity", "H1", ""),
    (r"era5_feels_like_(mean|max|min)", "°C", "Daily {0} feels-like temperature (NWS: heat index when hot and humid, wind chill when cold and windy, else air temperature)", "sensitivity", "H1", ""),
    (r"era5_rh_mean", "%", "Daily mean relative humidity, ERA5-Land", "secondary", "", "Adjustment candidate"),
    (r"era5_dewpoint_mean", "°C", "Daily mean dew point, ERA5-Land", "secondary", "", ""),
    (r"era5_wind_(mean|max)", "m/s", "Daily {0} wind speed at 10 m, ERA5", "secondary", "", ""),
    (r"era5_pressure_mean", "hPa", "Daily mean surface pressure, ERA5", "secondary", "", ""),
    # warnings
    (r"warn_date_assumed", "0/1", "All text warnings covering the day have an assumed or approximate date", "technical", "", ""),
    (r"warn_strict_(\w+)", "0/1", "ARSO text warning '{0}' with explicit date and an area clearly including Maribor", "exploratory", "", "Validated in 2.2.3"),
    (r"warn_(\w+)", "0/1", "ARSO text warning '{0}' whose area could include Maribor (broad)", "exploratory", "", "Misses most official warning days (warnings_validation.csv)"),
    (r"ma_warn_level", "0,2,3,4", "Highest official warning level active in Maribor (0 none, 2 yellow, 3 orange, 4 red)", "secondary", "", "Missing before MA_VALID_FROM (archive start)"),
    (r"ma_warn_any", "0/1", "Official warning (yellow or higher) active in Maribor", "secondary", "", "Missing before MA_VALID_FROM (archive start)"),
    (r"ma_warn_orange_plus", "0/1", "Official orange or red warning active in Maribor", "secondary", "", "Missing before MA_VALID_FROM (archive start)"),
    (r"ma_warn_(\w+)", "0,2,3,4", "Highest official {0} warning level active in Maribor", "secondary", "", "Missing before MA_VALID_FROM (archive start)"),
    # air
    (r"pm10", "µg/m³", "Daily mean PM10, station SI0059A Maribor Vrbanski plato (urban background), EEA verified daily values", "primary", "H3", "From 2026: unverified data from a different instrument (sampling point _101)"),
    (r"pm_unverified_instrument", "0/1", "1 = pm10/pm25 of that day come from hourly, unverified data of a different instrument at SI0059A (sampling point _101; 2026)", "technical", "",
     "Exclude or adjust in sensitivity analyses of H3"),
    (r"pm10_titova", "µg/m³", "Daily mean PM10, ARSO station MB Titova (traffic, city centre); day needs >= 18 valid hours", "sensitivity", "H3", "A different station from pm10: exposure-site sensitivity"),
    (r"pm10_vrbanski", "µg/m³", "Daily mean PM10, MB Vrbanski (same station as pm10, ARSO hourly data stream E408); day needs >= 18 valid hours", "sensitivity", "H3", "Same station as pm10, different data stream (r = 0.98): data-source sensitivity, not a second site"),
    (r"pm25", "µg/m³", "Daily mean PM2.5, station SI0059A Maribor Vrbanski plato (urban background)", "secondary", "H3 (secondary exposure)", "Analysed like H3, outside the Holm family of H1-H3"),
    (r"pm25_titova", "µg/m³", "Daily mean PM2.5, ARSO station MB Titova (traffic, city centre); day needs >= 18 valid hours", "sensitivity", "H3 (secondary exposure)", "A different station from pm25"),
    (r"pm25_vrbanski", "µg/m³", "Daily mean PM2.5, MB Vrbanski (same station as pm25, ARSO hourly data stream E408); day needs >= 18 valid hours", "sensitivity", "H3 (secondary exposure)", "Same station as pm25, different data stream"),
    (r"no2", "µg/m³", "Daily mean NO2, station SI0059A Maribor Vrbanski plato (urban background)", "secondary", "", ""),
    (r"aqi_max", "1-6", "Worst hourly European Air Quality Index level of the day (revised EEA bands; 1 good ... 6 extremely poor)", "secondary", "", "PM hourly from E408 (2021-2025)"),
    (r"aqi_mean", "1-6", "Mean hourly European Air Quality Index level of the day", "secondary", "", ""),
    (r"aqi_hours_poor", "hours", "Hours of the day with European Air Quality Index level 4 (poor) or worse", "secondary", "", ""),
    (r"aqi_n_hours", "hours", "Hours of the day with an index (>= 18 needed)", "technical", "", ""),
    (r"o3", "µg/m³", "Maximum daily 8-hour mean O3, station SI0059A Maribor Vrbanski plato (urban background)", "secondary", "", ""),
    # respiratory
    (r"ili_rate", "per 100,000", "ERVISS influenza-like illness consultation rate, Slovenia, all ages (ISO-week value)", "primary", "H2", "ERVISS starts mid-2022 (see valid_from)"),
    (r"ari_rate", "per 100,000", "ERVISS acute respiratory infection consultation rate, Slovenia (ISO-week value)", "secondary", "", ""),
    (r"pos_(flu|rsv|sars)", "%", "ERVISS sentinel positivity, {0} (ISO-week value)", "secondary", "", ""),
    (r"det_(flu|rsv|sars)", "detections/week", "ERVISS non-sentinel detections, {0} (ISO-week value)", "secondary", "", ""),
    (r"flu_index", "detections/week", "Combined weekly influenza activity: ERVISS influenza detections (Slovenia) from 2022-W25; before that Maribor PCR influenza positives scaled to the ERVISS level (ISO-week value)",
     "primary", "H1 (adjustment); H2 (sensitivity)", "Splice validated on the overlap weeks; no real-time version (forecasting: use det_flu_rt and lagged PCR)"),
    (r"flu_index_source", "0/1", "Source of flu_index: 1 = ERVISS, 0 = scaled Maribor PCR", "technical", "", ""),
    (r"pcr_(flu_a|flu_b)", "positives/day", "Positive PCR tests, {0}, Maribor (NLZOH), by date of receipt in the laboratory", "sensitivity", "H2",
     "Positives only, no test totals; dated by laboratory receipt, not sampling"),
    (r"pcr_(sars|rsv)", "positives/day", "Positive PCR tests, {0}, Maribor (NLZOH), by date of receipt in the laboratory", "exploratory", "",
     "Positives only, no test totals; dated by laboratory receipt, not sampling"),
    (r"covid_sewage_mb_age_days", "days", "Days since the wastewater sample used", "technical", "", ""),
    (r"covid_sewage_mb", "copies ratio", "SARS-CoV-2 N1 / PMMoV in Maribor wastewater, last sample carried forward", "exploratory", "", ""),
    (r"covid_cases_(mb|ce)", "cases/day", "New confirmed COVID-19 cases, region {0} (mb = Podravska, ce = Savinjska), by reporting date", "exploratory", "",
     "Depends on testing volume"),
    (r"covid_cases_stajerska", "cases/day", "New confirmed COVID-19 cases, Podravska + Savinjska, by reporting date", "exploratory", "",
     "Depends on testing volume"),
    (r"covid_active_stajerska", "cases", "Active COVID-19 cases, Podravska + Savinjska", "exploratory", "", ""),
    (r"covid_deaths_stajerska", "deaths/day", "New COVID-19 deaths, Podravska + Savinjska", "exploratory", "", ""),
    (r"covid_ukcmb_inpatients", "patients", "COVID-19 patients in hospital wards, UKC Maribor", "exploratory", "", ""),
    (r"covid_ukcmb_icu", "patients", "COVID-19 patients in intensive care, UKC Maribor", "exploratory", "", ""),
    (r"covid_ukcmb_admissions", "admissions/day", "New COVID-19 admissions, UKC Maribor", "exploratory", "", ""),
    (r"pollen_total", "grains/m³", "Daily pollen count, all taxa (as reported), Maribor", "exploratory", "", ""),
    (r"pollen_(\w+)", "grains/m³", "Daily pollen count, taxon {0}, Maribor", "exploratory", "", ""),
    (r"gt_(\w+)", "index 0-100", "Google Trends index for '{0}', monthly value", "descriptive", "",
     "Monthly and mostly 0 (low search volume, suppressed by Google): not informative as an exposure"),
    (r"sbi_return", "log return", "Daily log return of SBI TOP (trading days)", "secondary", "", "Outcome: arr_dx_R07"),
    (r"sbi_close", "index points", "SBI TOP closing value (trading days)", "descriptive", "", ""),
    (r"trading_day", "0/1", "Ljubljana Stock Exchange trading day", "technical", "", ""),
    (r"ndvi_(maribor|stajerska)_age_days", "days", "Days since the NDVI scene used ({0})", "technical", "", ""),
    (r"ndvi_(maribor|stajerska)", "NDVI", "Mean NDVI over {0}, last clear Sentinel-2 scene carried forward", "secondary", "",
     "Preregistered; nearly collinear with season"),
    (r"ems_calls", "calls/day", "EMS dispatch calls, Maribor", "exploratory", "", ""),
    (r"ems_vuz", "calls/day", "EMS calls with sni_vuz = DA", "exploratory", "", "Meaning of sni_vuz to document"),
    (r"football_matches", "matches/day", "Football matches, MNZ Maribor fixtures", "exploratory", "", ""),
    (r"hosp_occupancy", "patients", "Daily mean of the hourly total occupancy across departments, UKC Maribor", "exploratory", "",
     "Partly caused by ED arrivals: lagged use only"),
    (r"moon_illumination", "0-1", "Illuminated fraction of the moon at local noon", "exploratory", "", "Negative-control exposure"),
    (r"moon_age_days", "days", "Approximate days since the last new moon, at local noon (from the phase angle; error up to about 1 day)", "exploratory", "", "Negative-control exposure"),
    (r"moon_phase", "0-7", "Moon phase category (0 new ... 4 full ... 7 waning crescent)", "exploratory", "", "Negative-control exposure"),
    (r"full_moon", "0/1", "Local day of the full moon (US Naval Observatory times)", "exploratory", "", "Negative-control exposure"),
    (r"new_moon", "0/1", "Local day of the new moon (US Naval Observatory times)", "exploratory", "", "Negative-control exposure"),
    (r"full_moon_window", "0/1", "Full moon day +- 1 day", "exploratory", "", "Negative-control exposure"),
]
# 1. Availability for forecasting: days after the date until a value is known (origin = end of day t).
#    Negative = known before the day; NaN = not a real-time series (use the _rt version, or not for forecasting).
AVAILABILITY = {  # panel part -> (forecast_lag_days, description)
    "outcomes": (0, "end of day t (live hospital systems)"),
    "calendar": (-1, "known in advance"),
    "oxcgrt": (0, "policies in force on day t (dataset compiled later)"),
    "arso_weather": (1, "ARSO daily climate data, next day [assumption]"),
    "era5": (5, "ERA5 / ERA5-Land preliminary data, about 5 days behind real time"),
    "arso_warnings": (0, "issued on or before day t"),
    "meteoalarm": (0, "issued on or before day t"),
    "arso_air": (0, "end of day t (hourly near-real-time data)"),
    "erviss": (np.nan, "ISO-week value: not real-time (use the _rt version)"),
    "pcr": (1, "laboratory receipt date; result about 1 day later [assumption]"),
    "flu_index": (np.nan, "ISO-week value: not real-time"),
    "covid": (1, "Sledilnik daily reports, next day"),
    "pollen": (1, "next day [assumption]"),
    "google_trends": (np.nan, "monthly value: not real-time (use the _rt version)"),
    "sbi_top": (0, "end of trading day t (close 17:30)"),
    "ndvi": (np.nan, "carried from the scene date: use the _rt version"),
    "ems": (0, "end of day t"),
    "football": (-1, "fixtures published in advance"),
    "hospital_capacity": (0, "end of day t"),
    "moon": (-1, "known in advance"),
}


def availability_of(col: str, source: str) -> tuple:
    if col.endswith("_rt"):
        return 0, "aligned to publication: the value on day t was published by day t"
    if col.endswith("_interp"):
        return np.nan, "uses later values: never for forecasting"
    if col in ("covid_sewage_mb", "covid_sewage_mb_age_days"):
        return np.nan, "carried from the sampling date: use the _rt version"
    return AVAILABILITY[source]


USES = {"primary", "secondary", "sensitivity", "forecasting", "exploratory", "descriptive", "technical"}
VERSION_SUFFIX = {"_rt": ("real-time", "forecasting", "Real-time version: last value already published on that day, carried forward. "),
                  "_interp": ("interpolated", "sensitivity", "Interpolated version (linear between periods or samples; uses later values) of: ")}


def describe_column(col: str) -> dict:
    base, version, use_override, prefix = col, "main", None, ""
    for suffix, (v, u, n) in VERSION_SUFFIX.items():
        if col.endswith(suffix):
            base, version, use_override, prefix = col[: -len(suffix)], v, u, n
    for pattern, unit, definition, use, hyp, note in CODEBOOK_RULES:
        m = re.fullmatch(pattern, base)
        if m:
            groups = [STREAM_NAME.get(g) or NAME.get(g) or g.replace("_", " ") for g in m.groups()]
            return dict(unit=unit, definition=prefix + definition.format(*groups), version=version, use=use_override or use,
                        hypothesis="" if use_override == "forecasting" else hyp, note=note)
    raise KeyError(f"No codebook entry for column '{col}'")


codebook = pd.DataFrame([describe_column(c) for c in panel.columns], index=pd.Index(panel.columns, name="column"))
codebook.insert(0, "source", pd.Series(source_of).reindex(panel.columns))
avail_info = [availability_of(c, s) for c, s in zip(codebook.index, codebook["source"])]
codebook["forecast_lag_days"] = [a for a, _ in avail_info]
codebook["available"] = [t for _, t in avail_info]
codebook["valid_from"] = panel.apply(lambda s: s.first_valid_index()).dt.date
codebook["valid_to"] = panel.apply(lambda s: s.last_valid_index()).dt.date
codebook["pct_missing"] = (panel.isna().mean() * 100).round(1)
codebook["n_distinct"] = panel.nunique()
codebook["n_nonzero_days"] = (panel.fillna(0) != 0).sum()
assert codebook["use"].isin(USES).all(), codebook.loc[~codebook["use"].isin(USES), "use"]
assert codebook["source"].notna().all()
codebook.to_csv(PROCESSED / "daily_panel_codebook.csv")
print(f"Codebook: {len(codebook)} columns -> {(PROCESSED / 'daily_panel_codebook.csv').relative_to(ROOT)}")
display(codebook["use"].value_counts().rename("columns").to_frame())
print("Primary columns:")
codebook.loc[codebook["use"] == "primary", ["hypothesis", "unit", "definition", "valid_from", "valid_to", "pct_missing"]]

## 8. Reproducibility: data manifest and run record

`data/data_manifest.csv` (local only, gitignored with the data) lists every file under `data/raw/` with its size and SHA-256 checksum. The run **stops** if a raw file was added, removed or changed since the manifest was written, so a result can always be traced to an exact set of inputs. After a deliberate data update (new extract, refreshed download), set `MANIFEST_UPDATE = True` and run once.

`outputs/01_data_aggregation/run_info.json` records the run: date, git commit, Python and package versions, the manifest checksum, the panel checksum, and the date each downloaded cache was last written.

In [ ]:
current = raw_file_table()
if MANIFEST_UPDATE or not MANIFEST.exists():
    current.to_csv(MANIFEST, index=False)
    print(f"Manifest written: {len(current)} files, {current['bytes'].sum() / 1e6:.0f} MB -> {MANIFEST.relative_to(ROOT)}")
else:
    diff = compare_manifest(current)
    if any(diff.values()):
        hint = ""
        if any(p.startswith("meteoalarm/") for p in diff["missing"]):
            hint = (" The MeteoAlarm cache is missing: add METEOALARM_API_KEY to .env and run again, "
                    "or copy data/raw/meteoalarm/ from a colleague's checkout.")
        raise RuntimeError(f"Raw data differ from {MANIFEST.name}: {diff}.{hint} If the change is intended, set MANIFEST_UPDATE = True and run again.")
    print(f"All {len(current)} raw files match {MANIFEST.name} ({current['bytes'].sum() / 1e6:.0f} MB).")


def git(*args) -> str:
    try:
        return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
    except Exception:  # noqa: BLE001
        return "unavailable"


def last_written(folder: Path) -> str:
    t = max(p.stat().st_mtime for p in folder.rglob("*") if p.is_file())
    return pd.Timestamp(t, unit="s", tz="UTC").tz_convert("Europe/Ljubljana").isoformat(timespec="seconds")


CACHES = ["era5", "meteoalarm", "covid_sledilnik", "ndvi", "calendar", "moon"]
PACKAGES = ["pandas", "numpy", "pyarrow", "matplotlib", "plotly", "shapely", "requests", "python-calamine", "openpyxl",
            "odc-stac", "pystac-client", "planetary-computer"]
run_info = {
    "run_at": pd.Timestamp.now(tz="Europe/Ljubljana").isoformat(timespec="seconds"),
    "git_commit": git("rev-parse", "HEAD"),
    "git_uncommitted_changes": git("status", "--porcelain") not in ("", "unavailable"),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {p: pkg_version(p) for p in PACKAGES},
    "study_period": [str(START.date()), str(END.date())],
    "manifest_sha256": sha256(MANIFEST),
    "raw_files": len(current),
    "panel_shape": list(panel.shape),
    "panel_sha256": sha256(PROCESSED / "daily_panel.parquet"),
    "cache_last_written": {c: last_written(RAW / c) for c in CACHES if (RAW / c).exists()},
}
(OUTPUTS / "run_info.json").write_text(json.dumps(run_info, indent=1))
print(json.dumps({k: v for k, v in run_info.items() if k not in ("packages", "cache_last_written")}, indent=1))